# Poumons : masques pulmonaires, cartes mesurées dans les poumons et version v4 recadrée (notebook 16)

**Pourquoi ce notebook ?** Les notebooks 11 à 14 ont montré que les modèles ne regardent pas assez les poumons. En v2, une grande partie des cartes Grad-CAM tombe dans les bandes noires du padding. En v3, les cartes moyennes évitent les champs pulmonaires et s'allument sur les coins et sur le marqueur « R ». Jusqu'ici, le « centre » n'était qu'un rectangle fixe, pas une vraie zone pulmonaire. Ce notebook fait trois choses :

1. **Segmenter les poumons** de chaque radio avec un modèle public déjà entraîné : le PSPNet de la bibliothèque TorchXRayVision, entraîné sur le jeu ChestX-Det. La qualité des masques est ensuite contrôlée.
2. **Mesurer la part des cartes Grad-CAM qui tombe dans les poumons**, pour les 18 modèles existants (v2 sans augmentation, v2 avec augmentation, v3), avec les vrais contours pulmonaires à la place du rectangle central.
3. **Entraîner une version v4** : chaque radio est recadrée sur la boîte qui entoure les deux poumons, avec une marge de 10 %. Elle est ensuite préparée comme en v3 (étirement 224 × 224 puis égalisation). Le réseau ne voit presque plus que le thorax : les bords, les coins, le padding et la plupart des marqueurs sortent de l'entrée.

Le levier choisi agit sur **ce que le réseau voit**, sans changer l'architecture ni la perte. Les deux autres leviers pourront venir ensuite, avec les masques enregistrés ici : une perte qui guide l'attention vers les poumons, et un CBAM non résiduel.

## Protocole fixé le 28/09/2026, avant toute segmentation et tout entraînement v4

| Élément | Choix |
|---|---|
| Segmentation | PSPNet ChestX-Det (TorchXRayVision), poids contrôlés par SHA-256. Entrée : radio en niveaux de gris placée dans un carré 512 × 512 avec ses proportions (padding noir), valeurs ramenées dans [−1024 ; 1024] comme l'exige la bibliothèque. Sortie : probabilité des poumons droit et gauche, seuil 0,5. Pour chaque poumon, on garde la plus grande zone d'un seul tenant et on bouche les trous. |
| Masque valide | Chaque poumon couvre au moins 2 % de l'image ; les deux poumons réunis en couvrent au plus 70 % ; leurs centres sont écartés d'au moins 10 % de la largeur, dans l'ordre gauche/droite de la majorité des radios. Sinon le masque est invalide : la radio garde l'image entière en v4 (entrée identique à v3) et elle est exclue des mesures de cartes. |
| Recadrage v4 | Boîte qui entoure les deux poumons, agrandie de 10 % de sa hauteur en haut et en bas et de 10 % de sa largeur à gauche et à droite, sans sortir de l'image. Puis étirement en 224 × 224, égalisation d'histogramme et division par 255. |
| Entraînement v4 | Identique au notebook 12 : mêmes architectures, réglages, graines, optimiseur, poids des classes et budget (20 époques au plus, patience 4 sur le F1 macro de validation). |
| Cartes | Grad-CAM comme aux notebooks 11 et 13 (carte de décision de la classe prédite), sur la validation, pour les 24 modèles (18 existants et 6 v4). **Part dans les poumons** = masse de la carte dans les poumons ÷ masse totale de la carte. **Enrichissement pulmonaire** = cette part ÷ part de surface des poumons dans l'entrée. 1 = carte uniforme ; plus de 1 = carte concentrée dans les poumons. **Concentration pulmonaire** = (part dans les poumons − part de surface) ÷ (1 − part de surface) : 0 = carte uniforme, 1 = toute la carte dans les poumons, négative = carte qui évite les poumons. L'enrichissement est plafonné à 1 ÷ part de surface, plafond plus bas en v4 (les poumons y remplissent presque toute l'entrée) ; la concentration a les mêmes repères 0 et 1 dans toutes les préparations : c'est elle qui sert à comparer v3 et v4. Sensibilité : zone pulmonaire élargie de 8 pixels, au cas où le masque manquerait une partie d'une opacité. Seules les radios à masque valide entrent dans ces mesures. |

## Prédictions écrites le 28/09/2026, avant les résultats

| | Prédiction | Règle de lecture |
|---|---|---|
| P5 | Pour les décisions PNEUMONIA correctes, l'enrichissement pulmonaire de chacun des 6 modèles v4 est supérieur à 1. | confirmée si 6 cas sur 6 ; partielle si 4 ou 5 ; infirmée sinon |
| P6 | Pour les décisions PNEUMONIA correctes, la concentration pulmonaire est plus élevée en v4 qu'en v3, à architecture et graine égales. | même règle sur les 6 paires |
| P5 et P6 sensibilité | Mêmes prédictions avec la zone pulmonaire élargie de 8 pixels. | même règle |
| P7 | Parmi les radios PNEUMONIA, la dépendance du score au format d'origine (\|ρ\| de Spearman) est plus faible en v4 qu'en v3. | même règle sur les 6 paires |
| P8 | Le recadrage coûte au plus 0,02 de F1 macro de validation : F1(v4) ≥ F1(v3) − 0,02. | même règle sur les 6 paires |

**Amendement du 29/09/2026, avant tout lancement de ce notebook (aucun masque ni aucun résultat n'existait).** (1) P6 portait d'abord sur l'enrichissement pulmonaire. Elle porte sur la concentration pulmonaire, parce que l'enrichissement est plafonné à 1 ÷ part de surface : en v4 les poumons occupent une plus grande part de l'entrée, le plafond est donc plus bas et la comparaison aurait défavorisé v4 par construction. P5 est inchangée (concentration > 0 équivaut à enrichissement > 1). (2) Arrêt de sécurité : si moins de 50 % des masques sont valides dans une classe d'un ensemble, le notebook s'arrête juste après le contrôle de qualité, avant les cartes et l'entraînement, car un recadrage issu d'un segmenteur qui échoue massivement n'est pas interprétable. (3) Précision du 30/09/2026, toujours avant tout lancement : un modèle v4 qui ne prendrait aucune décision PNEUMONIA correcte sur la validation compterait comme cas défavorable pour P5, au lieu d'arrêter le notebook.

**Descriptif, sans verdict :** part des cartes dans les poumons pour les 18 modèles existants ; écart CBAM − classique en v4 (F1, AUC, spécificité, cartes) ; attention spatiale du CBAM dans et hors des poumons.

**Test : exploratoire seulement.** Le test a déjà servi au notebook 14, et l'idée de v4 vient en partie de ses résultats. Les modèles v4 y sont évalués pour décrire, avec les mêmes tirages par patient qu'au notebook 14, sans verdict. Les hypothèses H1 à H4 ne changent pas.

**Limites connues avant les résultats :** le segmenteur a été entraîné sur des radios d'adultes ; il peut échouer sur des enfants ou manquer des opacités denses. Le recadrage peut créer de nouveaux indices (taille ou proportions de la boîte) : c'est mesuré. Les réglages ne sont pas réoptimisés pour v4. Trois graines seulement.

**Exécution :** noyau `memoire_cnn`, cellules dans l'ordre. **À lancer le soir : compter 5 à 10 heures sur CPU.** La segmentation prend 1 h 30 à 5 h (une estimation s'affiche après le premier paquet) et l'entraînement des 6 modèles 3 à 6 h, comme au notebook 12. La première cellule installe TorchXRayVision une seule fois ; le premier passage télécharge les poids du segmenteur (273 Mo). Tout reprend après une interruption : la segmentation par paquets de 200 radios, les cartes modèle par modèle, l'entraînement époque par époque. **Garder OneDrive en pause, fermer les CSV dans Excel et empêcher la mise en veille.** À la fin, un seul fichier est à transmettre : `poumons_v4/a_envoyer_notebook_16.zip`.

In [ ]:
# Installation de TorchXRayVision (une seule fois), sans changer les bibliothèques déjà installées.
import importlib
import importlib.util
import subprocess
import sys
from importlib import metadata

A_CONSERVER = ["torch", "torchvision", "numpy", "pandas", "pillow", "scipy", "scikit-learn", "matplotlib", "tqdm", "requests"]

def versions_installees(noms):
    trouvees = {}
    for nom in noms:
        try:
            trouvees[nom] = metadata.version(nom)
        except metadata.PackageNotFoundError:
            pass
    return trouvees

if importlib.util.find_spec("torchxrayvision") is None:
    avant = versions_installees(A_CONSERVER)
    # Les versions déjà installées sont imposées à pip : il choisit des dépendances compatibles au lieu de les remplacer.
    commande = [sys.executable, "-m", "pip", "install", "torchxrayvision"] + [f"{nom}=={version}" for nom, version in avant.items()]
    print("Installation de torchxrayvision (quelques minutes). Versions conservées :", avant)
    resultat = subprocess.run(commande, capture_output=True, text=True)
    print("\n".join(resultat.stdout.strip().splitlines()[-8:]))
    if resultat.returncode != 0:
        print(resultat.stderr[-3000:])
        raise RuntimeError("L'installation a échoué : me transmettre le message ci-dessus.")
    apres = versions_installees(A_CONSERVER)
    changees = {nom: (avant[nom], apres.get(nom)) for nom in avant if apres.get(nom) != avant[nom]}
    if changees:
        raise RuntimeError(f"L'installation a modifié des bibliothèques : {changees}. Redémarrer le noyau et me transmettre ce message.")
    importlib.invalidate_caches()
print("torchxrayvision", metadata.version("torchxrayvision"), "est installé.")

In [ ]:
from pathlib import Path
from importlib import metadata
import hashlib
import json
import shutil
import sys
import time
import warnings
import zipfile
import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageOps
import scipy
from scipy import ndimage
from scipy.stats import rankdata, spearmanr
import torch
from torch import nn
import torch.nn.functional as F
from sklearn import __version__ as version_sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from tqdm.auto import tqdm
from IPython.display import display

# Si le projet change de dossier, modifier uniquement PROJET.
PROJET = Path(r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention")
IMAGES = PROJET / "archive" / "chest_xray"
REPARTITION = PROJET / "repartitions" / "repartition_20260917_102959_713272"
DOSSIER_08 = PROJET / "optimisation" / "robustesse_validation_v2"
DOSSIER_10 = PROJET / "optimisation" / "robustesse_augmentation_v2"
DOSSIER_12 = PROJET / "optimisation" / "v3_sans_raccourci_format"
DOSSIER_14 = PROJET / "evaluations" / "test_final_v2_v3"
SORTIE = PROJET / "optimisation" / "poumons_v4"
APPAREIL = torch.device("cpu")
TAILLE, LOT, TAILLE_CARTE, MAX_EPOQUES, PATIENCE = 224, 16, 28, 20, 4
RESOLUTION_SEGMENTATION = 512   # résolution native du segmenteur
TAILLE_MASQUE = 256             # masques enregistrés dans la géométrie de l'image entière, en 256 × 256
LOT_SEGMENTATION = 2            # radios segmentées ensemble (mémoire limitée)
TAILLE_PAQUET = 200             # radios par fichier de masques : reprise possible après une interruption
SEUIL_MASQUE = 0.5
AIRE_MIN_POUMON = 0.02          # chaque poumon couvre au moins 2 % de l'image
AIRE_MAX_POUMONS = 0.70         # les deux poumons réunis couvrent au plus 70 % de l'image
SEUIL_ARRET_MASQUES = 50.0      # arrêt de sécurité : moins de 50 % de masques valides dans une classe d'un ensemble
ECART_MIN_CENTRES = 0.10        # écart horizontal minimal entre les centres des deux poumons (part de la largeur)
MARGE_RECADRAGE = 0.10          # marge ajoutée de chaque côté de la boîte des poumons (part de sa taille)
DILATATION = 8                  # zone pulmonaire élargie de 8 pixels (à 224) : analyse de sensibilité
N_CONTROLE = 32                 # radios de validation qui servent à vérifier chaque modèle existant
N_BOOTSTRAP, GRAINE_BOOTSTRAP = 2000, 20260928   # mêmes tirages par patient qu'au notebook 14
POIDS_SEGMENTEUR = {"fichier": "pspnet_chestxray_best_model_4.pth", "octets": 272988989,
                    "sha256": "019b167eac6b729fc1bb92bbbc185fc1730aaa65819f4e3fe718186cadc044fc",
                    "url": "https://github.com/mlmed/torchxrayvision/releases/download/v1/pspnet_chestxray_best_model_4.pth"}
CLASSES = {"NORMAL": 0, "PNEUMONIA": 1}
NOMS_CLASSES = {0: "NORMAL", 1: "PNEUMONIA"}
GRAINES = [42, 123, 2026]
CONFIGURATIONS = {
    "CNN_classique": {"indice": 2, "lr": 0.001, "dropout": 0.5},
    "CNN_CBAM": {"indice": 3, "lr": 0.0005, "dropout": 0.3},
}
ARCHITECTURES_NOMS = ["CNN_classique", "CNN_CBAM"]
LIBELLES_MODELES = {"CNN_classique": "CNN classique", "CNN_CBAM": "CNN + CBAM"}
V2_SANS, V2_AVEC, V3, V4 = "v2 sans augmentation", "v2 avec augmentation", "v3 sans raccourci", "v4 recadrage poumons"
VERSIONS_EXISTANTES = [V2_SANS, V2_AVEC, V3]
VERSIONS_MODELES = [V2_SANS, V2_AVEC, V3, V4]
MODELES_EXISTANTS_ORDRE = [(v, m, g) for v in VERSIONS_EXISTANTES for m in ARCHITECTURES_NOMS for g in GRAINES]
PRETRAITEMENT_V2 = "L; ImageOps.pad 224 bilineaire noir centre; /255"
PRETRAITEMENTS = {
    V2_SANS: PRETRAITEMENT_V2 + "; aucune augmentation",
    V2_AVEC: PRETRAITEMENT_V2,
    V3: "L; resize 224x224 bilineaire sans padding (etirement); ImageOps.equalize; /255; aucune augmentation",
    V4: ("L; recadrage sur la boite des deux poumons (PSPNet ChestX-Det de torchxrayvision, seuil 0.5, plus grande composante "
         "et trous bouches par poumon, marge 10 %, image entiere si masque invalide); resize 224x224 bilineaire (etirement); "
         "ImageOps.equalize; /255; aucune augmentation"),
}
# Palette catégorielle des versions (notebooks 12 à 14), complétée par v4 ; écarts vérifiés pour les trois formes de daltonisme.
COULEURS = {V2_SANS: "#2a78d6", V2_AVEC: "#eb6834", V3: "#1baf7a", V4: "#b8428c"}
# Annotations dessinées sur les radios : contour des poumons et boîte de recadrage.
COULEUR_CONTOUR, COULEUR_BOITE = "#ffd23f", "#35d0ff"
ENCRE, ENCRE_SECONDAIRE, ENCRE_DISCRETE, TRAIT_ZERO = "#0b0b0b", "#52514e", "#898781", "#c3c2b7"
POSITIONS_MODELES = {"CNN_classique": 0, "CNN_CBAM": 1}

def version_de(bibliotheque):
    try:
        return metadata.version(bibliotheque)
    except metadata.PackageNotFoundError:
        return "absente"

VERSIONS = {
    "python": sys.version, "torch": str(torch.__version__), "numpy": np.__version__, "pandas": pd.__version__,
    "pillow": PIL.__version__, "sklearn": version_sklearn, "scipy": scipy.__version__,
    "torchxrayvision": version_de("torchxrayvision"),
}
torch.use_deterministic_algorithms(True)

def exiger(condition, message):
    if not condition:
        raise RuntimeError(message)

def empreinte_fichier(chemin):
    h = hashlib.sha256()
    with open(chemin, "rb") as fichier:
        for bloc in iter(lambda: fichier.read(1024 * 1024), b""):
            h.update(bloc)
    return h.hexdigest()

def remplacer(temporaire, chemin, essais=20, attente=1.5):
    # Windows ou OneDrive peuvent verrouiller un fichier quelques secondes : on réessaie.
    for essai in range(essais):
        try:
            temporaire.replace(chemin)
            return
        except PermissionError:
            if essai == essais - 1:
                raise PermissionError(f"Impossible d'écrire {chemin} : fermer ce fichier (Excel ?) "
                                      "ou suspendre OneDrive, puis relancer la cellule.")
            time.sleep(attente)

def ecrire_json(objet, chemin):
    temporaire = chemin.with_suffix(".tmp")
    temporaire.write_text(json.dumps(objet, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    remplacer(temporaire, chemin)

def ecrire_csv(tableau, chemin):
    temporaire = chemin.with_suffix(".tmp")
    tableau.to_csv(temporaire, index=False, encoding="utf-8-sig", float_format="%.17g")
    remplacer(temporaire, chemin)

def ecrire_npz(chemin, **tableaux):
    temporaire = chemin.with_suffix(".tmp")
    with open(temporaire, "wb") as fichier:
        np.savez_compressed(fichier, **tableaux)
    remplacer(temporaire, chemin)

def enregistrer_figure(figure, chemin):
    temporaire = chemin.with_suffix(".tmp")
    figure.savefig(temporaire, format="png", dpi=160, bbox_inches="tight")
    remplacer(temporaire, chemin)

def empreinte_objet(objet):
    return hashlib.sha256(json.dumps(objet, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()

def nom_fichier(version, nom, graine):
    prefixe = {V2_SANS: "v2_sans", V2_AVEC: "v2_avec", V3: "v3", V4: "v4"}[version]
    return f"{prefixe}__{nom}_graine_{graine}"

def dossier_v3(nom, graine):
    return DOSSIER_12 / nom / f"graine_{graine}" / f"configuration_{CONFIGURATIONS[nom]['indice']:02d}"

def dossier_v4(nom, graine):
    return SORTIE / "modeles_v4" / nom / f"graine_{graine}" / f"configuration_{CONFIGURATIONS[nom]['indice']:02d}"

def booleens(serie):
    valeurs = serie.map({True: True, False: False, "True": True, "False": False})
    exiger(valeurs.notna().all(), "Colonne booléenne inattendue.")
    return valeurs.astype(bool)

def lire_existant(fichier, lignes_attendues):
    """Résultats déjà calculés avec ce protocole, sinon None."""
    if not fichier.exists():
        return None
    tableau = pd.read_csv(fichier, dtype={"groupe": str})
    exiger(len(tableau) == lignes_attendues and (tableau["id_protocole"] == ID_PROTOCOLE).all(),
           f"{fichier.name} : résultats existants incompatibles avec ce protocole. Choisir un autre dossier SORTIE.")
    return tableau

def resumer_modeles(tableau, groupes, mesures):
    """Entrée : une ligne par graine. Sortie : moyenne, minimum et maximum sur les graines."""
    resume = tableau.groupby(groupes, sort=False)[mesures].agg(["mean", "min", "max"])
    resume.columns = [f"{mesure}_{statistique}" for mesure, statistique in resume.columns]
    return resume.reset_index()

def styliser(axe):
    axe.grid(axis="y", color="#e5e5e5", linewidth=0.8)
    axe.set_axisbelow(True)
    for cote in ["top", "right"]:
        axe.spines[cote].set_visible(False)

def points_par_graine(axe, tableau, colonne, versions):
    """Un point par graine et un trait pour la moyenne ; couleur = version, position = architecture."""
    decalages = dict(zip(versions, np.linspace(-0.27, 0.27, len(versions)) if len(versions) > 1 else [0.0]))
    for version in versions:
        for nom, x in POSITIONS_MODELES.items():
            y = tableau[(tableau["version"] == version) & (tableau["modele"] == nom)][colonne].to_numpy(dtype=float)
            y = y[np.isfinite(y)]
            if len(y) == 0:
                continue
            position = x + decalages[version]
            axe.scatter(np.full(len(y), position), y, s=64, color=COULEURS[version], edgecolors="white",
                        linewidths=1.5, zorder=3, label=version if nom == "CNN_classique" else None)
            axe.hlines(y.mean(), position - 0.07, position + 0.07, color=COULEURS[version], linewidth=2, zorder=2)
    axe.set_xticks(list(POSITIONS_MODELES.values()), [LIBELLES_MODELES[m] for m in POSITIONS_MODELES])
    axe.set_xlim(-0.5, 1.5)
    styliser(axe)

for dossier in [SORTIE, SORTIE / "masques", SORTIE / "cartes", SORTIE / "predictions_validation",
                SORTIE / "predictions_test", SORTIE / "figures"]:
    dossier.mkdir(parents=True, exist_ok=True)
print("Appareil :", APPAREIL, "| PyTorch :", torch.__version__, "| torchxrayvision :", VERSIONS["torchxrayvision"])
print("Résultats dans :", SORTIE)

## 1. Données, modèles existants et contrôles

Les trois ensembles sont relus et contrôlés (effectifs, aucun patient ni aucune image en commun). La validation est préparée en v2 et en v3, et le test en v3. Chaque préparation doit retrouver exactement l'empreinte SHA-256 enregistrée par les notebooks 10, 12 et 14.

Les 18 modèles existants sont rechargés : les 12 modèles v2 du notebook 10 v2 (fichiers contrôlés par empreinte) et les 6 modèles v3 du notebook 12. Avant d'enregistrer le protocole, chacun doit retrouver ses scores déjà enregistrés sur 32 radios de validation, et les modèles v3 doivent retrouver leurs scores du test du notebook 14.

In [ ]:
def preparer_v2(image):
    """Préparation v2 (notebooks 08 à 14) : proportions conservées, bandes noires, masque du rectangle de l'image."""
    gris = image.convert("L")
    preparee = ImageOps.pad(gris, (TAILLE, TAILLE), method=Image.Resampling.BILINEAR, color=0, centering=(0.5, 0.5))
    blanc = Image.new("L", gris.size, color=255)
    masque = ImageOps.pad(blanc, (TAILLE, TAILLE), method=Image.Resampling.NEAREST, color=0, centering=(0.5, 0.5))
    return np.array(preparee, dtype=np.uint8), np.array(masque) > 0

def etirer(image):
    """Première étape v3 (notebook 12) : niveaux de gris, étirement en 224 × 224 sans bandes noires."""
    return np.array(image.convert("L").resize((TAILLE, TAILLE), Image.Resampling.BILINEAR), dtype=np.uint8)

def egaliser(tableau_u8):
    """Seconde étape v3 (notebook 12) : égalisation d'histogramme."""
    return np.array(ImageOps.equalize(Image.fromarray(np.ascontiguousarray(tableau_u8, dtype=np.uint8))), dtype=np.uint8)

def rectangle_image(masque):
    """Rectangle occupé par l'image dans l'entrée v2, en indices inclus (r0, r1, c0, c1)."""
    lignes = np.flatnonzero(masque.any(axis=1))
    colonnes = np.flatnonzero(masque.any(axis=0))
    exiger(len(lignes) > 0 and len(colonnes) > 0, "Rectangle d'image vide.")
    r0, r1, c0, c1 = int(lignes[0]), int(lignes[-1]), int(colonnes[0]), int(colonnes[-1])
    exiger(bool(masque[r0:r1 + 1, c0:c1 + 1].all()) and int(masque.sum()) == (r1 - r0 + 1) * (c1 - c0 + 1),
           "La zone de l'image n'est pas un rectangle plein.")
    return r0, r1, c0, c1

# Résultats nécessaires des notebooks précédents.
for fichier, message in [(DOSSIER_10 / "termine.json", "Le notebook 10 v2 n'est pas terminé."),
                         (DOSSIER_12 / "resultats_v3.csv", "Le notebook 12 n'est pas terminé : resultats_v3.csv est absent."),
                         (DOSSIER_14 / "termine.json", "Le notebook 14 n'est pas terminé.")]:
    exiger(fichier.is_file(), message)
PROTOCOLE_10 = json.loads((DOSSIER_10 / "protocole.json").read_text(encoding="utf-8"))
exiger(PROTOCOLE_10.get("version_analyse") == "comparaison_robustesse_augmentation_v2"
       and PROTOCOLE_10.get("configurations") == CONFIGURATIONS, "Protocole du notebook 10 v2 inattendu.")
PROTOCOLE_14 = json.loads((DOSSIER_14 / "protocole.json").read_text(encoding="utf-8"))
exiger(PROTOCOLE_14.get("version_analyse") == "evaluation_finale_test_v1", "Protocole du notebook 14 inattendu.")
ID_PROTOCOLE_14 = empreinte_objet(PROTOCOLE_14)

# Répartition (notebook 02) : trois ensembles disjoints.
lecture = {"dtype": {"groupe": str, "empreinte": str}}
train_df = pd.read_csv(REPARTITION / "train.csv", **lecture)
val_df = pd.read_csv(REPARTITION / "val.csv", **lecture)
test_df = pd.read_csv(REPARTITION / "test.csv", **lecture)
ENSEMBLES = {"train": train_df, "val": val_df, "test": test_df}
EFFECTIFS_ATTENDUS = {"train": {"NORMAL": 1079, "PNEUMONIA": 2721}, "val": {"NORMAL": 269, "PNEUMONIA": 684},
                      "test": {"NORMAL": 231, "PNEUMONIA": 387}}
for ensemble, tableau in ENSEMBLES.items():
    exiger({"chemin_relatif", "groupe", "empreinte", "classe"}.issubset(tableau.columns), f"Colonnes absentes : {ensemble}.")
    exiger(tableau["classe"].value_counts().to_dict() == EFFECTIFS_ATTENDUS[ensemble], f"Effectifs inattendus : {ensemble}.")
    exiger(tableau["chemin_relatif"].is_unique and tableau["groupe"].notna().all(), f"Chemins dupliqués ou patients inconnus : {ensemble}.")
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    for colonne in ["groupe", "empreinte", "chemin_relatif"]:
        exiger(set(ENSEMBLES[a][colonne]).isdisjoint(ENSEMBLES[b][colonne]), f"{a} et {b} partagent des valeurs de « {colonne} ».")
SHA_CSV = {ensemble: empreinte_fichier(REPARTITION / f"{ensemble}.csv") for ensemble in ENSEMBLES}
exiger(SHA_CSV["val"] == PROTOCOLE_10["sha256_val_csv"], "La validation diffère de celle du notebook 10 v2.")
exiger(SHA_CSV["test"] == PROTOCOLE_14["sha256_test_csv"], "Le test diffère de celui du notebook 14.")
Y = {ensemble: tableau["classe"].map(CLASSES).to_numpy(dtype=np.int64) for ensemble, tableau in ENSEMBLES.items()}
IDENTIFIANTS, REELS, N_VAL = val_df["chemin_relatif"].tolist(), Y["val"], len(val_df)
IDENTIFIANTS_TEST, Y_TEST, N_TEST = test_df["chemin_relatif"].tolist(), Y["test"], len(test_df)

# Modèles v3 du notebook 12.
RESULTATS_V3 = pd.read_csv(DOSSIER_12 / "resultats_v3.csv")
exiger(len(RESULTATS_V3) == 6 and set(zip(RESULTATS_V3["modele"], RESULTATS_V3["graine"].astype(int)))
       == {(m, g) for m in ARCHITECTURES_NOMS for g in GRAINES}, "resultats_v3.csv doit contenir les six modèles v3.")
for ligne in RESULTATS_V3.to_dict("records"):
    reglages = CONFIGURATIONS[ligne["modele"]]
    # Tolérance : la relecture d'un CSV peut modifier le dernier chiffre d'un nombre décimal.
    exiger(int(ligne["indice"]) == reglages["indice"] and abs(float(ligne["lr"]) - reglages["lr"]) < 1e-12
           and abs(float(ligne["dropout"]) - reglages["dropout"]) < 1e-12,
           f"Réglages v3 inattendus : {ligne['modele']}, graine {ligne['graine']}.")
PROTOCOLES_V3 = {}
for nom in ARCHITECTURES_NOMS:
    for graine in GRAINES:
        fichier = dossier_v3(nom, graine) / "protocole.json"
        exiger(fichier.is_file(), f"Protocole v3 introuvable : {fichier}")
        protocole = json.loads(fichier.read_text(encoding="utf-8"))
        exiger(protocole.get("pretraitement") == PRETRAITEMENTS[V3] and protocole.get("graine") == graine
               and protocole.get("sha256_csv") == {"train": SHA_CSV["train"], "val": SHA_CSV["val"]},
               f"Protocole v3 inattendu : {fichier}")
        PROTOCOLES_V3[(nom, graine)] = protocole

# Validation préparée en v2 et en v3, identique aux notebooks 08 à 13.
PREPAREES = np.empty((N_VAL, TAILLE, TAILLE), dtype=np.uint8)
MASQUES = np.empty((N_VAL, TAILLE, TAILLE), dtype=bool)
EGALISEES = np.empty((N_VAL, TAILLE, TAILLE), dtype=np.uint8)
LARGEURS = np.empty(N_VAL, dtype=np.int64)
HAUTEURS = np.empty(N_VAL, dtype=np.int64)
BORNES = []
empreinte_v2, empreinte_v3 = hashlib.sha256(), hashlib.sha256()
for i, identifiant in enumerate(tqdm(IDENTIFIANTS, desc="Préparation v2 et v3 de la validation")):
    with Image.open(IMAGES / identifiant) as image:
        LARGEURS[i], HAUTEURS[i] = image.size
        PREPAREES[i], MASQUES[i] = preparer_v2(image)
        EGALISEES[i] = egaliser(etirer(image))
    exiger(MASQUES[i].any() and (PREPAREES[i][~MASQUES[i]] == 0).all(), "Erreur de géométrie du padding.")
    BORNES.append(rectangle_image(MASQUES[i]))
    empreinte_v2.update(identifiant.encode("utf-8") + b"\0")
    empreinte_v2.update(PREPAREES[i].tobytes())
    empreinte_v2.update(MASQUES[i].tobytes())
    empreinte_v3.update(identifiant.encode("utf-8") + b"\0")
    empreinte_v3.update(EGALISEES[i].tobytes())
SHA_IMAGES_V2, SHA_V3 = empreinte_v2.hexdigest(), empreinte_v3.hexdigest()
exiger(SHA_IMAGES_V2 == PROTOCOLE_10["sha256_images_preparees_et_masques"], "Les images v2 diffèrent de celles des notebooks 08 à 13.")
for (nom, graine), protocole in PROTOCOLES_V3.items():
    exiger(protocole["sha256_entrees_v3"]["val"] == SHA_V3, f"Les entrées v3 diffèrent de celles du notebook 12 ({nom}, graine {graine}).")
RAPPORTS = LARGEURS / HAUTEURS

# Test préparé en v3, identique au notebook 14.
EGALISEES_TEST = np.empty((N_TEST, TAILLE, TAILLE), dtype=np.uint8)
RAPPORTS_TEST = np.empty(N_TEST, dtype=np.float64)
empreinte = hashlib.sha256()
for i, identifiant in enumerate(tqdm(IDENTIFIANTS_TEST, desc="Préparation v3 du test")):
    with Image.open(IMAGES / identifiant) as image:
        largeur, hauteur = image.size
        EGALISEES_TEST[i] = egaliser(etirer(image))
    RAPPORTS_TEST[i] = largeur / hauteur
    empreinte.update(identifiant.encode("utf-8") + b"\0")
    empreinte.update(EGALISEES_TEST[i].tobytes())
SHA_TEST_V3 = empreinte.hexdigest()
exiger(SHA_TEST_V3 == PROTOCOLE_14["sha256_entrees_test_v3"], "Les entrées v3 du test diffèrent de celles du notebook 14.")

def fichier_reference(version, nom, graine):
    """Prédictions de validation déjà enregistrées (notebooks 08 v2, 10 v2 et 12)."""
    if version == V2_SANS:
        return DOSSIER_08 / "predictions" / f"{nom}_graine_{graine}__originale.csv"
    if version == V2_AVEC:
        return DOSSIER_10 / "predictions" / f"avec__{nom}_graine_{graine}__originale.csv"
    return DOSSIER_12 / "predictions_validation" / f"{nom}_graine_{graine}.csv"

def logits_reference(version, nom, graine):
    chemin = fichier_reference(version, nom, graine)
    exiger(chemin.is_file(), f"Prédictions de référence introuvables : {chemin}")
    tableau = pd.read_csv(chemin, dtype={"groupe": str}, float_precision="round_trip")
    exiger(tableau["chemin_relatif"].tolist() == IDENTIFIANTS, f"Ordre des images différent : {chemin}")
    return tableau[["logit_NORMAL", "logit_PNEUMONIA"]].to_numpy(dtype=np.float64)

REFERENCES = {cle: logits_reference(*cle) for cle in MODELES_EXISTANTS_ORDRE}
for ligne in RESULTATS_V3.to_dict("records"):
    logits = REFERENCES[(V3, ligne["modele"], int(ligne["graine"]))]
    f1 = f1_score(REELS, (logits[:, 1] > logits[:, 0]).astype(int), labels=[0, 1], average="macro", zero_division=0)
    exiger(abs(f1 - float(ligne["val_f1_macro"])) < 1e-10,
           f"Prédictions v3 incohérentes avec resultats_v3.csv : {ligne['modele']}, graine {ligne['graine']}.")

# Scores des modèles v3 sur le test (notebook 14), pour la partie exploratoire.
PREDICTIONS_TEST_V3 = {}
for nom in ARCHITECTURES_NOMS:
    for graine in GRAINES:
        chemin = DOSSIER_14 / "predictions" / f"{nom_fichier(V3, nom, graine)}.csv"
        exiger(chemin.is_file(), f"Prédictions du test introuvables : {chemin}")
        tableau = pd.read_csv(chemin, dtype={"groupe": str}, float_precision="round_trip")
        exiger(len(tableau) == N_TEST and (tableau["id_protocole"] == ID_PROTOCOLE_14).all()
               and tableau["chemin_relatif"].tolist() == IDENTIFIANTS_TEST, f"Prédictions du notebook 14 inattendues : {chemin}")
        PREDICTIONS_TEST_V3[(nom, graine)] = tableau

def entrees_val(version, indices):
    """Entrées du réseau (lot numpy de forme (B, 1, 224, 224)) pour des radios de validation."""
    if version in (V2_SANS, V2_AVEC):
        source = PREPAREES
    elif version == V3:
        source = EGALISEES
    else:
        source = IMAGES_V4["val"]
    return (source[np.asarray(indices)].astype(np.float32) / 255.0)[:, None]

def controler_logits(calcules, attendus, contexte):
    ecart = float(np.abs(np.asarray(calcules, dtype=np.float64) - attendus).max())
    tolerance = 1e-4 + 1e-5 * float(np.abs(attendus).max())
    exiger(ecart <= tolerance, f"{contexte} : scores bruts différents des prédictions de référence (écart maximal {ecart:.2e}). "
                               "Vérifier le noyau, les versions et les sauvegardes avant de continuer.")
    return ecart

print(f"Apprentissage {len(train_df)}, validation {N_VAL}, test {N_TEST} radios ; aucun patient en commun.")
print("Préparations v2 et v3 identiques à celles des notebooks 08 à 14 (empreintes contrôlées).")

In [ ]:
class CNNClassique(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc3 = nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.reduction = nn.AdaptiveAvgPool2d((4,4))
        self.classificateur = nn.Sequential(nn.Flatten(),nn.Linear(64*4*4,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,2))
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(x))

class AttentionCanaux(nn.Module):
    def __init__(self, canaux, reduction=16):
        super().__init__()
        dim=max(1,canaux//reduction)
        self.reseau_partage=nn.Sequential(
            nn.Conv2d(canaux,dim,1,bias=False),nn.ReLU(),
            nn.Conv2d(dim,canaux,1,bias=False))
    def forward(self,x):
        moyenne=x.mean(dim=(2,3),keepdim=True)
        maximum=x.amax(dim=(2,3),keepdim=True)
        return x*torch.sigmoid(self.reseau_partage(moyenne)+self.reseau_partage(maximum))

class AttentionSpatiale(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolution=nn.Conv2d(2,1,7,padding=3,bias=False)
    def forward(self,x):
        resume=torch.cat([x.mean(dim=1,keepdim=True),x.amax(dim=1,keepdim=True)],dim=1)
        return x*torch.sigmoid(self.convolution(resume))

class CBAM(nn.Module):
    def __init__(self,canaux=64,reduction=16):
        super().__init__()
        self.attention_canaux=AttentionCanaux(canaux,reduction)
        self.attention_spatiale=AttentionSpatiale()
    def forward(self,x):
        # Connexion résiduelle, identique aux notebooks 06 v2 à 14.
        attenue = self.attention_spatiale(self.attention_canaux(x))
        return x + attenue

class CNNAvecCBAM(CNNClassique):
    def __init__(self,dropout=0.3):
        super().__init__(dropout)
        self.cbam=CBAM(64,16)
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(self.cbam(x)))

ARCHITECTURES = {
    "CNN_classique": (CNNClassique, "CNNClassique_89026_v1", 89026),
    "CNN_CBAM": (CNNAvecCBAM, "CNNAvecCBAM_89636_apres_bloc3_r16_k7_residuel_v2", 89636),
}
VERSION_DE_VARIANTE = {"sans": V2_SANS, "avec": V2_AVEC}

def charger_modele(version, nom, graine, chemin):
    classe, architecture, nombre_parametres = ARCHITECTURES[nom]
    etat = torch.load(chemin, map_location="cpu", weights_only=True)
    protocole = etat["protocole"]
    exiger(protocole["architecture"] == architecture, f"Architecture différente : {version}, {nom}.")
    exiger(protocole["graine"] == graine, f"Graine incompatible : {version}, {nom}, {graine}.")
    exiger(protocole["classes"] == CLASSES and protocole["pretraitement"] == PRETRAITEMENTS[version],
           f"Classes ou préparation incompatibles : {version}, {nom}, graine {graine}.")
    exiger(etat["configuration"] == CONFIGURATIONS[nom], f"Sauvegarde d'une autre configuration : {version}, {nom}, graine {graine}.")
    if version == V3:
        exiger(protocole["sha256_entrees_v3"]["val"] == SHA_V3, f"Entrées v3 différentes de celles du notebook 12 : {nom}, graine {graine}.")
    if version == V4:
        exiger(protocole["sha256_entrees_v4"]["val"] == SHA_V4["val"], f"Entrées v4 différentes : {nom}, graine {graine}.")
    with torch.random.fork_rng(devices=[]):
        modele = classe(dropout=CONFIGURATIONS[nom]["dropout"]).to(APPAREIL)
        modele.load_state_dict(etat["modele"], strict=True)
    exiger(sum(p.numel() for p in modele.parameters()) == nombre_parametres, "Nombre de paramètres inattendu.")
    modele.eval()
    modele.requires_grad_(False)
    return modele, int(etat["epoque"])

MODELES, INFORMATIONS = {}, []
exiger(len(PROTOCOLE_10["modeles"]) == 12, "Le protocole du notebook 10 v2 doit lister douze modèles.")
for info in PROTOCOLE_10["modeles"]:
    version, nom, graine = VERSION_DE_VARIANTE[info["variante"]], info["modele"], int(info["graine"])
    chemin = PROJET.joinpath(*info["sauvegarde_relative"].split("/"))
    exiger(chemin.is_file(), f"Sauvegarde introuvable : {chemin}")
    exiger(empreinte_fichier(chemin) == info["sha256_sauvegarde"], f"La sauvegarde {chemin} a changé depuis le notebook 10 v2.")
    modele, epoque = charger_modele(version, nom, graine, chemin)
    exiger(epoque == int(info["epoque"]), "Époque incompatible avec le notebook 10 v2.")
    MODELES[(version, nom, graine)] = modele
    INFORMATIONS.append({"version": version, "modele": nom, "graine": graine, "epoque": epoque,
                         "sha256_sauvegarde": info["sha256_sauvegarde"], "sauvegarde_relative": info["sauvegarde_relative"]})
for ligne in RESULTATS_V3.to_dict("records"):
    nom, graine = ligne["modele"], int(ligne["graine"])
    chemin = dossier_v3(nom, graine) / "meilleur.pt"
    exiger(chemin.is_file(), f"Sauvegarde v3 introuvable : {chemin}")
    modele, epoque = charger_modele(V3, nom, graine, chemin)
    exiger(epoque == int(ligne["epoque_retenue"]), f"Époque v3 incompatible avec resultats_v3.csv : {nom}, graine {graine}.")
    MODELES[(V3, nom, graine)] = modele
    INFORMATIONS.append({"version": V3, "modele": nom, "graine": graine, "epoque": epoque,
                         "sha256_sauvegarde": empreinte_fichier(chemin), "sauvegarde_relative": chemin.relative_to(PROJET).as_posix()})
exiger(set(MODELES) == set(MODELES_EXISTANTS_ORDRE), "Il manque des modèles.")

def logits_lot(modele, x):
    """Logits (B, 2) pour un lot numpy de forme (B, 1, 224, 224)."""
    with torch.no_grad():
        sortie = modele(torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)).to(APPAREIL))
    return sortie.cpu().numpy().astype(np.float64)

class Capture:
    """Conserve la sortie d'une couche pendant le passage avant du modèle."""
    def __init__(self, module):
        self.sortie = None
        self.poignee = module.register_forward_hook(self.enregistrer)
    def enregistrer(self, module, entree, sortie):
        self.sortie = sortie
    def retirer(self):
        self.poignee.remove()

def calculer_lot(nom, modele, x):
    """Logits (B, 2), carte Grad-CAM signée (B, 28, 28) et attention spatiale (B, 28, 28) ou None, pour un lot numpy."""
    couche = modele.cbam if nom == "CNN_CBAM" else modele.bloc3
    capture = Capture(couche)
    capture_spatiale = Capture(modele.cbam.attention_spatiale.convolution) if nom == "CNN_CBAM" else None
    try:
        entree = torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)).to(APPAREIL).requires_grad_(True)
        with torch.enable_grad():
            logits = modele(entree)
            activation = capture.sortie
            exiger(activation is not None and tuple(activation.shape) == (x.shape[0], 64, TAILLE_CARTE, TAILLE_CARTE),
                   "La couche utilisée pour Grad-CAM n'a pas la forme attendue.")
            difference = logits[:, 1] - logits[:, 0]
            (gradient,) = torch.autograd.grad(difference.sum(), activation)
        poids = gradient.mean(dim=(2, 3), keepdim=True)
        carte_signee = (poids * activation.detach()).sum(dim=1)
        spatiale = None
        if capture_spatiale is not None:
            sortie = capture_spatiale.sortie
            exiger(sortie is not None and tuple(sortie.shape) == (x.shape[0], 1, TAILLE_CARTE, TAILLE_CARTE),
                   "L'attention spatiale n'a pas la forme attendue.")
            spatiale = torch.sigmoid(sortie.detach())[:, 0].cpu().numpy()
    finally:
        capture.retirer()
        if capture_spatiale is not None:
            capture_spatiale.retirer()
    logits = logits.detach().cpu().numpy().astype(np.float64)
    carte_signee = carte_signee.cpu().numpy()
    exiger(np.isfinite(logits).all() and np.isfinite(carte_signee).all(), "Valeurs non finies dans les logits ou la carte.")
    return logits, carte_signee, spatiale

def agrandir(cartes):
    """Agrandit des cartes (B, 28, 28) en (B, 224, 224) par interpolation bilinéaire."""
    t = torch.from_numpy(np.ascontiguousarray(cartes, dtype=np.float32)).unsqueeze(1)
    return F.interpolate(t, size=(TAILLE, TAILLE), mode="bilinear", align_corners=False)[:, 0].numpy().astype(np.float64)

print("18 modèles existants chargés et figés : 12 modèles v2 (notebook 10 v2) et 6 modèles v3 (notebook 12).")

In [ ]:
# Contrôles AVANT d'enregistrer le protocole : chaque modèle existant retrouve ses scores déjà enregistrés.
ECARTS_CONTROLE = {}
for cle in MODELES_EXISTANTS_ORDRE:
    logits = np.concatenate([logits_lot(MODELES[cle], entrees_val(cle[0], np.arange(d, min(d + LOT, N_CONTROLE))))
                             for d in range(0, N_CONTROLE, LOT)])
    ECARTS_CONTROLE[cle] = controler_logits(logits, REFERENCES[cle][:N_CONTROLE], f"{nom_fichier(*cle)}, contrôle sur la validation")
x_test_v3 = (EGALISEES_TEST.astype(np.float32) / 255.0)[:, None]
ECARTS_TEST_V3 = {}
for nom in ARCHITECTURES_NOMS:
    for graine in GRAINES:
        logits = np.concatenate([logits_lot(MODELES[(V3, nom, graine)], x_test_v3[d:d + LOT]) for d in range(0, N_TEST, LOT)])
        attendus = PREDICTIONS_TEST_V3[(nom, graine)][["logit_NORMAL", "logit_PNEUMONIA"]].to_numpy(dtype=np.float64)
        ECARTS_TEST_V3[(nom, graine)] = controler_logits(logits, attendus, f"v3, {nom}, graine {graine}, test du notebook 14")
print(f"Contrôle réussi : 18 modèles sur 32 radios de validation (écart maximal {max(ECARTS_CONTROLE.values()):.1e}) ; "
      f"modèles v3 sur le test du notebook 14 (écart maximal {max(ECARTS_TEST_V3.values()):.1e}).")

PREDICTIONS = {
    "P5": "Pour les décisions PNEUMONIA correctes, l'enrichissement pulmonaire de chacun des 6 modèles v4 est supérieur à 1 (carte plus concentrée dans les poumons qu'une carte uniforme).",
    "P5 sensibilité": "Même prédiction que P5, avec la zone pulmonaire élargie de 8 pixels.",
    "P6": "Pour les décisions PNEUMONIA correctes, la concentration pulmonaire est plus élevée en v4 qu'en v3, à architecture et graine égales.",
    "P6 sensibilité": "Même comparaison que P6, avec la zone pulmonaire élargie de 8 pixels.",
    "P7": "Parmi les radios PNEUMONIA, la dépendance du score au format d'origine (|ρ| de Spearman) est plus faible en v4 qu'en v3, à architecture et graine égales.",
    "P8": "Le recadrage coûte au plus 0,02 de F1 macro de validation : F1(v4) ≥ F1(v3) − 0,02, à architecture et graine égales.",
}
ORDRE_BILAN = list(PREDICTIONS)
REGLE = "confirmée si 6 cas favorables sur 6 ; partielle si 4 ou 5 ; infirmée sinon"

def verdict(favorables, total):
    exiger(total == 6, "Chaque prédiction porte sur 6 cas.")
    if favorables == total:
        return "confirmée"
    return "partielle" if favorables >= 4 else "infirmée"

PROTOCOLE = {
    "version_analyse": "poumons_v4",
    "predictions_ecrites_le_28_09_2026": PREDICTIONS, "regle_de_lecture": REGLE,
    "amendement_29_09_2026": "avant tout lancement : P6 portee par la concentration pulmonaire (l'enrichissement est plafonne a 1/aire, plus bas en v4); arret de securite si moins de 50 % de masques valides dans une classe d'un ensemble",
    "segmentation": {
        "modele": "torchxrayvision.baseline_models.chestx_det.PSPNet (ChestX-Det, Lian et al. 2021)",
        "poids": POIDS_SEGMENTEUR, "resolution": RESOLUTION_SEGMENTATION,
        "entree": "L; ImageOps.pad 512x512 bilineaire noir centre; x/255*2048-1024",
        "sortie": "sigmoide des canaux Right Lung et Left Lung; zone de l'image (hors padding) redimensionnee en 256x256 (bilineaire): geometrie de l'image entiere",
        "seuil": SEUIL_MASQUE, "nettoyage": "par poumon : plus grande composante connexe (8-connexite), trous bouches",
        "validite": {"aire_min_par_poumon": AIRE_MIN_POUMON, "aire_max_deux_poumons": AIRE_MAX_POUMONS,
                     "ecart_min_centres": ECART_MIN_CENTRES,
                     "lateralite": "ordre gauche/droite de la majorite des radios dont les deux poumons sont presents"},
        "paquets": TAILLE_PAQUET, "lot": LOT_SEGMENTATION, "arret_securite_masques_valides_pct": SEUIL_ARRET_MASQUES,
        "fragilite": "resultats v4 signales comme fragiles si moins de 90 % de masques valides dans une classe d'un ensemble",
    },
    "recadrage_v4": {"marge": MARGE_RECADRAGE,
                     "boite": "boite des deux poumons nettoyes, agrandie de la marge (part de sa largeur et de sa hauteur, de chaque cote), limitee a l'image",
                     "masque_invalide": "image entiere : entree identique a v3"},
    "pretraitement_v4": PRETRAITEMENTS[V4],
    "entrainement_v4": {"comme": "notebook 12", "configurations": CONFIGURATIONS, "graines": GRAINES, "lot": LOT,
                        "max_epoques": MAX_EPOQUES, "patience": PATIENCE, "critere": "val_f1_macro"},
    "cartes": {"methode": "Grad-CAM a la sortie de bloc3 (classique) ou du CBAM residuel (CBAM), 64x28x28; score logit PNEUMONIA - logit NORMAL; carte de decision = ReLU de la carte signee de la classe predite; agrandissement bilineaire 224x224",
               "part_poumons": "masse dans la zone pulmonaire / masse totale de la carte (padding compris pour v2)",
               "enrichissement": "part_poumons / part de surface de la zone pulmonaire dans l'entree 224x224; moyenne sur les radios",
               "concentration": "(part_poumons - aire) / (1 - aire), aire = part de surface de la zone pulmonaire dans l'entree 224x224; 0 = carte uniforme, 1 = tout dans les poumons; sert a P6 et P6 sensibilite",
               "sensibilite": f"zone pulmonaire elargie de {DILATATION} pixels (dilatation carree), limitee a l'image",
               "radios": "validation, masques valides seulement; decisions PNEUMONIA correctes pour P5 et P6",
               "attention_CBAM": "moyenne de l'attention spatiale dans la zone pulmonaire et dans le reste de l'image"},
    "format": "rho de Spearman entre le rapport largeur/hauteur d'origine et le score (logit PNEUMONIA - logit NORMAL), radios PNEUMONIA de la validation",
    "test_exploratoire": {"statut": "exploratoire, sans verdict : test deja utilise au notebook 14",
                          "bootstrap": {"tirages": N_BOOTSTRAP, "graine": GRAINE_BOOTSTRAP, "comme": "notebook 14 (memes tirages par patient)"},
                          "comparaisons": "v4 - v3 par architecture et pour les deux; CBAM - classique en v3 et en v4"},
    "sha256_csv": SHA_CSV, "sha256_images_v2_val": SHA_IMAGES_V2, "sha256_entrees_v3_val": SHA_V3,
    "sha256_entrees_v3_test": SHA_TEST_V3, "id_protocole_notebook_14": ID_PROTOCOLE_14,
    "modeles_existants": INFORMATIONS, "versions": VERSIONS, "threads_torch": torch.get_num_threads(),
    "limite": "Segmenteur entraine sur des adultes; opacites denses parfois hors masque; nouveaux indices possibles lies au recadrage; reglages non reoptimises pour v4; validation deja utilisee pour choisir les epoques; test deja consulte (notebook 14).",
}
ID_PROTOCOLE = empreinte_objet(PROTOCOLE)
exiger(set(ECARTS_CONTROLE) == set(MODELES_EXISTANTS_ORDRE) and len(ECARTS_TEST_V3) == 6,
       "Les contrôles des modèles doivent réussir avant d'enregistrer le protocole.")
fichier_protocole = SORTIE / "protocole.json"
if fichier_protocole.exists():
    exiger(json.loads(fichier_protocole.read_text(encoding="utf-8")) == PROTOCOLE,
           "Le protocole, les modèles, les images ou les versions des bibliothèques ont changé. "
           "Conserver le dossier existant et choisir un autre nom SORTIE.")
else:
    ecrire_json(PROTOCOLE, fichier_protocole)
BILAN = {}
print("Protocole enregistré :", fichier_protocole)

## 2. Segmenter les poumons de toutes les radios

Les 5 371 radios (apprentissage, validation, test) sont segmentées une seule fois. Le segmenteur reçoit la radio en niveaux de gris, placée dans un carré 512 × 512 avec ses proportions. Il renvoie une probabilité par pixel pour le poumon droit et pour le poumon gauche du patient. La zone de l'image (sans le padding) est ramenée en 256 × 256 et enregistrée : c'est la **géométrie de l'image entière**, à partir de laquelle on obtient le masque dans chaque préparation (v2, v3, v4).

Les masques sont écrits par paquets de 200 radios dans `masques/`. Après une interruption, les paquets terminés sont reconnus. **C'est l'étape la plus longue avant l'entraînement** : le temps restant s'affiche après chaque paquet.

In [ ]:
import torchxrayvision as xrv

# La bibliothèque utilise encore F.upsample, dont PyTorch signale l'obsolescence à chaque appel : message masqué.
warnings.filterwarnings("ignore", message=r".*upsample.*")
warnings.filterwarnings("ignore", message=r".*align_corners.*")
chemin_poids = Path(xrv.utils.get_cache_dir()) / POIDS_SEGMENTEUR["fichier"]
if chemin_poids.is_file() and chemin_poids.stat().st_size != POIDS_SEGMENTEUR["octets"]:
    print("Le fichier de poids est incomplet (téléchargement interrompu) : il est supprimé puis téléchargé à nouveau.")
    chemin_poids.unlink()
if not chemin_poids.is_file():
    print("Téléchargement des poids du segmenteur (273 Mo), une seule fois :", POIDS_SEGMENTEUR["url"])
with torch.random.fork_rng(devices=[]):
    SEGMENTEUR = xrv.baseline_models.chestx_det.PSPNet()
SEGMENTEUR.eval()
SEGMENTEUR.requires_grad_(False)
exiger(chemin_poids.is_file() and empreinte_fichier(chemin_poids) == POIDS_SEGMENTEUR["sha256"],
       f"Poids du segmenteur différents de ceux prévus : supprimer {chemin_poids} puis relancer la cellule.")
CIBLES = list(SEGMENTEUR.targets)
exiger({"Right Lung", "Left Lung"}.issubset(CIBLES), "Le segmenteur ne fournit pas les poumons droit et gauche.")
INDICES_POUMONS = [CIBLES.index("Right Lung"), CIBLES.index("Left Lung")]

def segmenter_lot(x):
    """x : tableau (B, 512, 512) dans [-1024, 1024]. Renvoie les probabilités (B, 2, 512, 512) : poumon droit, poumon gauche du patient."""
    with torch.inference_mode():
        sortie = SEGMENTEUR(torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32))[:, None])
        exiger(tuple(sortie.shape) == (x.shape[0], len(CIBLES), RESOLUTION_SEGMENTATION, RESOLUTION_SEGMENTATION),
               "Sortie du segmenteur inattendue.")
        return torch.sigmoid(sortie[:, INDICES_POUMONS]).float().numpy()

print("Segmenteur prêt :", SEGMENTEUR, "| structures :", len(CIBLES), "| poids :", chemin_poids)

In [ ]:
TOUTES = pd.concat([tableau.assign(ensemble=ensemble)[["ensemble", "chemin_relatif", "groupe", "classe"]]
                    for ensemble, tableau in ENSEMBLES.items()], ignore_index=True)
N_TOTAL = len(TOUTES)
POSITION = {chemin: k for k, chemin in enumerate(TOUTES["chemin_relatif"])}
PAQUETS = [(debut, min(debut + TAILLE_PAQUET, N_TOTAL)) for debut in range(0, N_TOTAL, TAILLE_PAQUET)]
STRUCTURE = np.ones((3, 3), dtype=bool)

def boite_contenu(taille, cote):
    """Rectangle (gauche, haut, droite, bas) occupé par l'image dans ImageOps.pad(image, (cote, cote)) ; droite et bas exclus."""
    blanc = ImageOps.pad(Image.new("L", taille, 255), (cote, cote), method=Image.Resampling.NEAREST, color=0, centering=(0.5, 0.5))
    return blanc.getbbox()

def entree_segmentation(image):
    """Radio en niveaux de gris dans un carré 512 × 512 (proportions conservées, padding noir), valeurs dans [-1024, 1024]."""
    gris = image.convert("L")
    carree = ImageOps.pad(gris, (RESOLUTION_SEGMENTATION, RESOLUTION_SEGMENTATION), method=Image.Resampling.BILINEAR,
                          color=0, centering=(0.5, 0.5))
    x = np.asarray(carree, dtype=np.float32) / 255.0 * 2048.0 - 1024.0
    return x, boite_contenu(gris.size, RESOLUTION_SEGMENTATION)

def geometrie_relative(probabilite, boite):
    """Carte 512 × 512 (avec padding) → zone de l'image seule, en 256 × 256 (géométrie de l'image entière), codée de 0 à 255."""
    carte = Image.fromarray(np.ascontiguousarray(probabilite, dtype=np.float32))
    reduite = carte.resize((TAILLE_MASQUE, TAILLE_MASQUE), Image.Resampling.BILINEAR, box=tuple(float(v) for v in boite))
    return np.rint(np.clip(np.asarray(reduite, dtype=np.float32), 0.0, 1.0) * 255.0).astype(np.uint8)

def fichier_paquet(k):
    return SORTIE / "masques" / f"paquet_{k:03d}.npz"

def paquet_fait(k):
    debut, fin = PAQUETS[k]
    if not fichier_paquet(k).exists():
        return False
    with np.load(fichier_paquet(k)) as donnees:
        exiger(str(donnees["id_protocole"]) == ID_PROTOCOLE
               and donnees["chemins"].tolist() == TOUTES["chemin_relatif"].iloc[debut:fin].tolist(),
               f"{fichier_paquet(k).name} : masques incompatibles avec ce protocole. Choisir un autre dossier SORTIE.")
    return True

def segmenter_paquet(k):
    debut, fin = PAQUETS[k]
    chemins = TOUTES["chemin_relatif"].iloc[debut:fin].tolist()
    droit = np.empty((fin - debut, TAILLE_MASQUE, TAILLE_MASQUE), dtype=np.uint8)
    gauche = np.empty_like(droit)
    for d in tqdm(range(0, fin - debut, LOT_SEGMENTATION), desc=f"Paquet {k + 1}/{len(PAQUETS)}", leave=False):
        indices = list(range(d, min(d + LOT_SEGMENTATION, fin - debut)))
        entrees, boites = [], []
        for j in indices:
            with Image.open(IMAGES / chemins[j]) as image:
                x, boite = entree_segmentation(image)
            entrees.append(x)
            boites.append(boite)
        probabilites = segmenter_lot(np.stack(entrees))
        exiger(probabilites.shape == (len(indices), 2, RESOLUTION_SEGMENTATION, RESOLUTION_SEGMENTATION)
               and bool(np.isfinite(probabilites).all()), "Probabilités de segmentation invalides.")
        for j, p, boite in zip(indices, probabilites, boites):
            droit[j] = geometrie_relative(p[0], boite)
            gauche[j] = geometrie_relative(p[1], boite)
    ecrire_npz(fichier_paquet(k), chemins=np.array(chemins), droit=droit, gauche=gauche, id_protocole=np.array(ID_PROTOCOLE))

restants = [k for k in range(len(PAQUETS)) if not paquet_fait(k)]
print(f"Segmentation : {len(PAQUETS) - len(restants)} paquets sur {len(PAQUETS)} déjà faits.")
debut_segmentation, faites = time.perf_counter(), 0
for rang, k in enumerate(restants):
    segmenter_paquet(k)
    faites += PAQUETS[k][1] - PAQUETS[k][0]
    ecoule = time.perf_counter() - debut_segmentation
    reste = sum(PAQUETS[j][1] - PAQUETS[j][0] for j in restants[rang + 1:]) * ecoule / faites
    print(f"Paquet {k + 1}/{len(PAQUETS)} terminé | {ecoule / faites:.1f} s par radio | "
          f"reste environ {int(reste // 3600)} h {int(reste % 3600 // 60):02d} min")

def nettoyer(probabilite_u8):
    """Poumon binaire (probabilité ≥ 0,5) : plus grande zone d'un seul tenant, trous bouchés."""
    binaire = probabilite_u8 >= 128
    etiquettes, nombre = ndimage.label(binaire, structure=STRUCTURE)
    if nombre == 0:
        return binaire, 0, np.nan
    tailles = np.bincount(etiquettes.ravel())[1:]
    principale = etiquettes == int(np.argmax(tailles)) + 1
    return ndimage.binary_fill_holes(principale), int(nombre), float(tailles.max() / tailles.sum())

def qualite_masque(droit_u8, gauche_u8):
    """Mesures de qualité d'un masque (géométrie de l'image entière, coordonnées entre 0 et 1) et zone pulmonaire nettoyée."""
    droit, n_droit, principale_droit = nettoyer(droit_u8)
    gauche, n_gauche, principale_gauche = nettoyer(gauche_u8)
    zone = droit | gauche
    surface = float(TAILLE_MASQUE * TAILLE_MASQUE)
    centres = (np.arange(TAILLE_MASQUE) + 0.5) / TAILLE_MASQUE
    def centre(masque, axe):
        return float(centres[np.nonzero(masque)[axe]].mean()) if masque.any() else np.nan
    ligne = {"aire_poumon_droit": float(droit.sum()) / surface, "aire_poumon_gauche": float(gauche.sum()) / surface,
             "aire_poumons": float(zone.sum()) / surface, "composantes_droit": n_droit, "composantes_gauche": n_gauche,
             "part_principale_droit": principale_droit, "part_principale_gauche": principale_gauche,
             "centre_x_droit": centre(droit, 1), "centre_y_droit": centre(droit, 0),
             "centre_x_gauche": centre(gauche, 1), "centre_y_gauche": centre(gauche, 0)}
    if zone.any():
        lignes, colonnes = np.flatnonzero(zone.any(axis=1)), np.flatnonzero(zone.any(axis=0))
        ligne.update({"poumons_x0": colonnes[0] / TAILLE_MASQUE, "poumons_x1": (colonnes[-1] + 1) / TAILLE_MASQUE,
                      "poumons_y0": lignes[0] / TAILLE_MASQUE, "poumons_y1": (lignes[-1] + 1) / TAILLE_MASQUE})
    else:
        ligne.update({"poumons_x0": np.nan, "poumons_x1": np.nan, "poumons_y0": np.nan, "poumons_y1": np.nan})
    return ligne, zone

def boite_recadrage(ligne):
    """Boîte v4 (x0, y0, x1, y1) en parts de l'image, et indication d'une boîte limitée par le bord de l'image."""
    if not ligne["masque_valide"]:
        return 0.0, 0.0, 1.0, 1.0, False
    x0, x1, y0, y1 = ligne["poumons_x0"], ligne["poumons_x1"], ligne["poumons_y0"], ligne["poumons_y1"]
    mx, my = MARGE_RECADRAGE * (x1 - x0), MARGE_RECADRAGE * (y1 - y0)
    boite = (x0 - mx, y0 - my, x1 + mx, y1 + my)
    touche = boite[0] < 0 or boite[1] < 0 or boite[2] > 1 or boite[3] > 1
    return max(0.0, boite[0]), max(0.0, boite[1]), min(1.0, boite[2]), min(1.0, boite[3]), bool(touche)

lignes = []
POSITION_VAL = {chemin: i for i, chemin in enumerate(IDENTIFIANTS)}
ZONES_REL_VAL = np.zeros((N_VAL, TAILLE_MASQUE, TAILLE_MASQUE), dtype=bool)
for k, (debut, fin) in enumerate(tqdm(PAQUETS, desc="Contrôle de qualité des masques")):
    with np.load(fichier_paquet(k)) as donnees:
        droit, gauche = donnees["droit"], donnees["gauche"]
    for j, info in enumerate(TOUTES.iloc[debut:fin].to_dict("records")):
        ligne, zone = qualite_masque(droit[j], gauche[j])
        lignes.append({**info, **ligne})
        if info["ensemble"] == "val":
            ZONES_REL_VAL[POSITION_VAL[info["chemin_relatif"]]] = zone
QUALITE = pd.DataFrame(lignes)

# Latéralité : ordre gauche/droite de la majorité des radios dont les deux poumons sont présents.
presents = (QUALITE["aire_poumon_droit"] >= AIRE_MIN_POUMON) & (QUALITE["aire_poumon_gauche"] >= AIRE_MIN_POUMON)
exiger(bool(presents.any()), "Aucune radio n'a ses deux poumons détectés : vérifier le segmenteur avant de continuer.")
ecart_brut = QUALITE["centre_x_gauche"] - QUALITE["centre_x_droit"]
SENS_LATERALITE = 1.0 if float(np.median(ecart_brut[presents])) >= 0 else -1.0
QUALITE["ecart_centres"] = ecart_brut * SENS_LATERALITE
raisons = pd.DataFrame({
    "poumon droit absent ou trop petit": QUALITE["aire_poumon_droit"] < AIRE_MIN_POUMON,
    "poumon gauche absent ou trop petit": QUALITE["aire_poumon_gauche"] < AIRE_MIN_POUMON,
    "poumons trop étendus": QUALITE["aire_poumons"] > AIRE_MAX_POUMONS,
    "centres inversés ou trop proches": presents & ~(QUALITE["ecart_centres"] >= ECART_MIN_CENTRES),
})
QUALITE["masque_valide"] = ~raisons.any(axis=1)
QUALITE["raison_invalide"] = [" ; ".join(r for r, v in zip(raisons.columns, l) if v) for l in raisons.to_numpy()]
boites = pd.DataFrame([boite_recadrage(l) for l in QUALITE.to_dict("records")], index=QUALITE.index,
                      columns=["recadrage_x0", "recadrage_y0", "recadrage_x1", "recadrage_y1", "boite_touche_bord"])
QUALITE = pd.concat([QUALITE, boites], axis=1)
QUALITE_INDEX = QUALITE.set_index("chemin_relatif")
ecrire_csv(QUALITE, SORTIE / "qualite_masques.csv")
print(f"Masques valides : {100 * QUALITE['masque_valide'].mean():.1f} % des {N_TOTAL} radios. "
      f"Convention du segmenteur : le poumon gauche du patient est à {'droite' if SENS_LATERALITE > 0 else 'gauche'} de l'image.")

In [ ]:
RESUME_QUALITE = (QUALITE.groupby(["ensemble", "classe"], sort=False)
                  .agg(radios=("chemin_relatif", "size"),
                       masques_valides_pct=("masque_valide", lambda s: 100 * float(s.mean())),
                       aire_poumons_moyenne=("aire_poumons", "mean"),
                       aire_poumons_p05=("aire_poumons", lambda s: float(s.quantile(0.05))),
                       boite_touche_bord_pct=("boite_touche_bord", lambda s: 100 * float(s.astype(bool).mean())),
                       composantes_droit_moyenne=("composantes_droit", "mean"),
                       composantes_gauche_moyenne=("composantes_gauche", "mean"))
                  .reset_index())
RAISONS = (QUALITE[~QUALITE["masque_valide"]].groupby(["ensemble", "classe", "raison_invalide"], sort=False)
           .size().rename("radios").reset_index())
FRAGILE = bool((RESUME_QUALITE["masques_valides_pct"] < 90).any())
ecrire_csv(RESUME_QUALITE, SORTIE / "resume_qualite_masques.csv")
ecrire_csv(RAISONS, SORTIE / "raisons_masques_invalides.csv")
print("Qualité des masques par ensemble et par classe (aires en part de l'image) :")
display(RESUME_QUALITE.round(3))
print("Raisons des masques invalides :")
display(RAISONS)
if FRAGILE:
    print("ATTENTION : moins de 90 % de masques valides dans au moins une classe d'un ensemble. "
          "Règle fixée à l'avance : les résultats v4 seront signalés comme fragiles.")

PAQUET_CHARGE = {"k": None}

def masques_bruts(chemin):
    """Probabilités (0 à 255) des poumons droit et gauche d'une radio, en géométrie de l'image entière."""
    position = POSITION[chemin]
    k = position // TAILLE_PAQUET
    if PAQUET_CHARGE["k"] != k:
        with np.load(fichier_paquet(k)) as donnees:
            PAQUET_CHARGE.update({"k": k, "droit": donnees["droit"], "gauche": donnees["gauche"]})
    return PAQUET_CHARGE["droit"][position - PAQUETS[k][0]], PAQUET_CHARGE["gauche"][position - PAQUETS[k][0]]

def vignette(chemin, cote=512):
    with Image.open(IMAGES / chemin) as image:
        gris = image.convert("L")
    gris.thumbnail((cote, cote))
    return np.asarray(gris)

def dessiner_masque(axe, chemin, titre):
    """Radio d'origine avec le contour de chaque poumon (D = droit, G = gauche du patient) et la boîte de recadrage v4."""
    image = vignette(chemin)
    hauteur, largeur = image.shape
    axe.imshow(image, cmap="gray", extent=(0, largeur, hauteur, 0))
    xs = (np.arange(TAILLE_MASQUE) + 0.5) / TAILLE_MASQUE * largeur
    ys = (np.arange(TAILLE_MASQUE) + 0.5) / TAILLE_MASQUE * hauteur
    for probabilite, lettre in zip(masques_bruts(chemin), ["D", "G"]):
        poumon = nettoyer(probabilite)[0]
        if poumon.any():
            axe.contour(xs, ys, poumon.astype(float), levels=[0.5], colors=[COULEUR_CONTOUR], linewidths=1.3)
            lignes_p, colonnes_p = np.nonzero(poumon)
            axe.text(xs[int(round(colonnes_p.mean()))], ys[int(round(lignes_p.mean()))], lettre, fontsize=8, color=ENCRE,
                     ha="center", va="center", bbox={"facecolor": COULEUR_CONTOUR, "edgecolor": "none", "pad": 1.2})
    ligne = QUALITE_INDEX.loc[chemin]
    x0, y0, x1, y1 = (float(ligne[c]) for c in ["recadrage_x0", "recadrage_y0", "recadrage_x1", "recadrage_y1"])
    axe.add_patch(Rectangle((x0 * largeur, y0 * hauteur), (x1 - x0) * largeur, (y1 - y0) * hauteur, fill=False,
                            edgecolor=COULEUR_BOITE, linewidth=1.3, linestyle="--"))
    axe.set_xlim(0, largeur)
    axe.set_ylim(hauteur, 0)
    axe.set_title(titre, fontsize=8)
    axe.axis("off")

def titre_masque(chemin):
    ligne = QUALITE_INDEX.loc[chemin]
    etat = "masque valide" if ligne["masque_valide"] else "invalide : " + ligne["raison_invalide"].split(" ; ")[0]
    return f"{ligne['ensemble']} · {ligne['classe']} · poumons {100 * ligne['aire_poumons']:.0f} %\n{etat}"

LEGENDE_MASQUES = ("Contour jaune : poumons (D = droit du patient, G = gauche) ; pointillés bleu clair : boîte de recadrage v4 "
                   "(image entière si le masque est invalide)")
choix = (val_df[val_df["classe"] == "NORMAL"]["chemin_relatif"].head(6).tolist()
         + val_df[val_df["classe"] == "PNEUMONIA"]["chemin_relatif"].head(6).tolist())
fig, axes = plt.subplots(3, 4, figsize=(13, 11))
for axe, chemin in zip(axes.ravel(), choix):
    dessiner_masque(axe, chemin, titre_masque(chemin))
fig.suptitle("Contrôle visuel des masques : 6 premières radios NORMAL et 6 premières radios PNEUMONIA de la validation\n" + LEGENDE_MASQUES)
fig.tight_layout()
enregistrer_figure(fig, SORTIE / "figures" / "masques_exemples_validation.png")
plt.show()
plt.close(fig)

pires = QUALITE.nsmallest(12, "aire_poumons")["chemin_relatif"].tolist()
fig, axes = plt.subplots(3, 4, figsize=(13, 11))
for axe, chemin in zip(axes.ravel(), pires):
    dessiner_masque(axe, chemin, titre_masque(chemin))
for axe in axes.ravel()[len(pires):]:
    axe.axis("off")
fig.suptitle("Les 12 radios dont la zone pulmonaire détectée est la plus petite (tous ensembles) : échecs possibles du segmenteur\n"
             + LEGENDE_MASQUES)
fig.tight_layout()
enregistrer_figure(fig, SORTIE / "figures" / "masques_plus_petits.png")
plt.show()
plt.close(fig)

# Arrêt de sécurité fixé à l'avance (amendement du 29/09/2026), placé après les figures pour qu'elles soient disponibles.
ARRET = RESUME_QUALITE[RESUME_QUALITE["masques_valides_pct"] < SEUIL_ARRET_MASQUES]
exiger(ARRET.empty, f"Arrêt de sécurité : moins de {SEUIL_ARRET_MASQUES:.0f} % de masques valides dans {ARRET[['ensemble', 'classe', 'masques_valides_pct']].round(1).to_dict('records')}. Regarder les figures masques_*.png et me les transmettre : le segmenteur ne convient probablement pas à ces radios.")
print("Contrôle de qualité passé : au moins", f"{SEUIL_ARRET_MASQUES:.0f} % de masques valides dans chaque classe de chaque ensemble.")


## 3. Où tombent les cartes des 18 modèles existants ?

Les cartes Grad-CAM sont recalculées sur la validation, avec le même code qu'au notebook 13. Chaque modèle doit retrouver ses scores bruts déjà enregistrés. La zone pulmonaire de chaque radio est placée dans la géométrie de l'entrée du modèle :

- **v2** : image réduite avec ses proportions et bandes noires autour ; le masque est placé dans le rectangle de l'image ;
- **v3** : image entière étirée en 224 × 224 ;
- **v4** (plus loin) : boîte des poumons étirée en 224 × 224.

Pour chaque radio à masque valide : **part dans les poumons** (masse de la carte dans les poumons ÷ masse totale, padding compris pour v2), **part de surface des poumons** dans l'entrée, et leur rapport, l'**enrichissement pulmonaire** (1 = carte uniforme). Pour le CBAM : valeur moyenne de l'attention spatiale dans les poumons et dans le reste de l'image. Un fichier par modèle est écrit dans `cartes/` ; après une interruption, les modèles déjà traités sont réutilisés.

In [ ]:
def redimensionner_masque(masque, taille, boite=None):
    """Masque booléen vers une autre géométrie : interpolation bilinéaire de l'image 0/255, puis seuil à 128."""
    image = Image.fromarray(masque.astype(np.uint8) * 255)
    return np.asarray(image.resize(taille, Image.Resampling.BILINEAR, box=boite)) >= 128

def zone_v2(zone_relative, i):
    r0, r1, c0, c1 = BORNES[i]
    resultat = np.zeros((TAILLE, TAILLE), dtype=bool)
    resultat[r0:r1 + 1, c0:c1 + 1] = redimensionner_masque(zone_relative, (c1 - c0 + 1, r1 - r0 + 1))
    return resultat

def zone_v4(zone_relative, boite):
    x0, y0, x1, y1 = boite
    return redimensionner_masque(zone_relative, (TAILLE, TAILLE),
                                 (x0 * TAILLE_MASQUE, y0 * TAILLE_MASQUE, x1 * TAILLE_MASQUE, y1 * TAILLE_MASQUE))

def elargir(zone, contenu):
    return ndimage.binary_dilation(zone, structure=STRUCTURE, iterations=DILATATION) & contenu

CONTENU_PLEIN = np.ones((TAILLE, TAILLE), dtype=bool)
VALIDES_VAL = QUALITE_INDEX.loc[IDENTIFIANTS, "masque_valide"].to_numpy(dtype=bool)
BOITES_VAL = QUALITE_INDEX.loc[IDENTIFIANTS, ["recadrage_x0", "recadrage_y0", "recadrage_x1", "recadrage_y1"]].to_numpy(dtype=np.float64)
ZONES_V2 = np.stack([zone_v2(ZONES_REL_VAL[i], i) for i in range(N_VAL)])
ZONES_V2_ELARGIES = np.stack([elargir(ZONES_V2[i], MASQUES[i]) for i in range(N_VAL)])
ZONES_V3 = np.stack([redimensionner_masque(ZONES_REL_VAL[i], (TAILLE, TAILLE)) for i in range(N_VAL)])
ZONES_V3_ELARGIES = np.stack([elargir(zone, CONTENU_PLEIN) for zone in ZONES_V3])
ZONES_V4 = np.stack([zone_v4(ZONES_REL_VAL[i], tuple(BOITES_VAL[i])) for i in range(N_VAL)])
ZONES_V4_ELARGIES = np.stack([elargir(zone, CONTENU_PLEIN) for zone in ZONES_V4])

def geometrie(version, i):
    """Zone pulmonaire, zone élargie et partie de l'entrée occupée par l'image, dans la géométrie de la version."""
    if version in (V2_SANS, V2_AVEC):
        return ZONES_V2[i], ZONES_V2_ELARGIES[i], MASQUES[i]
    if version == V3:
        return ZONES_V3[i], ZONES_V3_ELARGIES[i], CONTENU_PLEIN
    return ZONES_V4[i], ZONES_V4_ELARGIES[i], CONTENU_PLEIN

def mesures_carte(carte, zone, elargie, contenu):
    total = float(carte.sum())
    if not np.isfinite(total) or total <= 1e-12:
        return {"part_poumons": np.nan, "part_poumons_elargis": np.nan, "part_hors_image": np.nan, "carte_decision_nulle": True}
    return {"part_poumons": float(carte[zone].sum()) / total, "part_poumons_elargis": float(carte[elargie].sum()) / total,
            "part_hors_image": float(carte[~contenu].sum()) / total, "carte_decision_nulle": False}

def cumuler(sommes, cle, carte):
    if cle not in sommes:
        sommes[cle] = [np.zeros((TAILLE, TAILLE), dtype=np.float64), 0]
    sommes[cle][0] += carte
    sommes[cle][1] += 1

def traiter_cartes(version, nom, graine):
    cle = nom_fichier(version, nom, graine)
    fichier_csv, fichier_npz = SORTIE / "cartes" / f"{cle}.csv", SORTIE / "cartes" / f"{cle}.npz"
    existant = lire_existant(fichier_csv, N_VAL) if fichier_npz.exists() else None
    if existant is not None:
        return existant, True
    modele, reference = MODELES[(version, nom, graine)], REFERENCES[(version, nom, graine)]
    logits = np.empty((N_VAL, 2), dtype=np.float64)
    sommes, lignes = {}, []
    vides = {"part_poumons": np.nan, "part_poumons_elargis": np.nan, "part_hors_image": np.nan, "carte_decision_nulle": np.nan}
    for debut in tqdm(range(0, N_VAL, LOT), desc=cle, leave=False):
        indices = np.arange(debut, min(debut + LOT, N_VAL))
        lg, signee, spatiale = calculer_lot(nom, modele, entrees_val(version, indices))
        logits[indices] = lg
        if debut == 0:
            controler_logits(lg, reference[indices], f"{cle}, cartes, premier lot")
        pneumonie, normale = agrandir(np.maximum(signee, 0.0)), agrandir(np.maximum(-signee, 0.0))
        attention = agrandir(spatiale) if spatiale is not None else None
        for k, i in enumerate(indices):
            zone, elargie, contenu = geometrie(version, i)
            prediction = int(lg[k, 1] > lg[k, 0])
            correcte = prediction == int(REELS[i])
            decision = pneumonie[k] if prediction == 1 else normale[k]
            valide = bool(VALIDES_VAL[i])
            m = mesures_carte(decision, zone, elargie, contenu) if valide else dict(vides)
            ligne = {"version": version, "modele": nom, "graine": graine, "chemin_relatif": IDENTIFIANTS[i],
                     "groupe": val_df["groupe"].iat[i], "classe_vraie": int(REELS[i]), "prediction": prediction,
                     "correcte": correcte, "difference_logits": float(lg[k, 1] - lg[k, 0]), "masque_valide": valide,
                     "aire_poumons": float(zone.mean()) if valide else np.nan,
                     "aire_poumons_elargis": float(elargie.mean()) if valide else np.nan, **m,
                     "attention_poumons": np.nan, "attention_hors_poumons": np.nan}
            if attention is not None and valide:
                dehors = contenu & ~zone
                ligne["attention_poumons"] = float(attention[k][zone].mean()) if zone.any() else np.nan
                ligne["attention_hors_poumons"] = float(attention[k][dehors].mean()) if dehors.any() else np.nan
                cumuler(sommes, f"attention__{int(REELS[i])}", attention[k])
            if valide and correcte and not m["carte_decision_nulle"]:
                cumuler(sommes, f"decision__{int(REELS[i])}", decision / decision.max())
            lignes.append(ligne)
    ecart = controler_logits(logits, reference, f"{cle}, cartes")
    tableau = pd.DataFrame(lignes)
    tableau["ecart_max_logits_reference"] = ecart
    tableau["id_protocole"] = ID_PROTOCOLE
    ecrire_npz(fichier_npz, **{f"{c}__somme": v[0].astype(np.float32) for c, v in sommes.items()},
               **{f"{c}__nombre": np.array(v[1]) for c, v in sommes.items()})
    ecrire_csv(tableau, fichier_csv)
    return tableau, False

def traiter_versions(cles):
    tableaux = []
    for cle in cles:
        debut = time.perf_counter()
        tableau, reutilise = traiter_cartes(*cle)
        tableaux.append(tableau)
        etat = "réutilisé" if reutilise else f"calculé en {(time.perf_counter() - debut) / 60:.1f} min"
        print(f"{cle[0]:<21} | {cle[1]:<13} | graine {cle[2]:>4} | écart aux logits de référence "
              f"{tableau['ecart_max_logits_reference'].max():.1e} | {etat}")
    return pd.concat(tableaux, ignore_index=True)

def preparer_cartes(tableau):
    """Radios à masque valide ; enrichissements calculés radio par radio."""
    t = tableau.copy()
    t["correcte"] = booleens(t["correcte"])
    t["masque_valide"] = booleens(t["masque_valide"])
    t = t[t["masque_valide"]].copy()
    t["carte_decision_nulle"] = booleens(t["carte_decision_nulle"]).astype(float)
    t["enrichissement_poumons"] = t["part_poumons"] / t["aire_poumons"]
    t["enrichissement_poumons_elargis"] = t["part_poumons_elargis"] / t["aire_poumons_elargis"]
    t["concentration_poumons"] = ((t["part_poumons"] - t["aire_poumons"]) / (1 - t["aire_poumons"])).where(t["aire_poumons"] < 1)
    t["concentration_poumons_elargis"] = ((t["part_poumons_elargis"] - t["aire_poumons_elargis"])
                                          / (1 - t["aire_poumons_elargis"])).where(t["aire_poumons_elargis"] < 1)
    t["ecart_attention_poumons"] = t["attention_poumons"] - t["attention_hors_poumons"]
    t["classe"] = t["classe_vraie"].map(NOMS_CLASSES)
    return t

MESURES_CARTES = ["aire_poumons", "part_poumons", "enrichissement_poumons", "concentration_poumons", "part_poumons_elargis",
                  "enrichissement_poumons_elargis", "concentration_poumons_elargis", "part_hors_image", "carte_decision_nulle"]
CARTES_EXISTANTES = traiter_versions(MODELES_EXISTANTS_ORDRE)
exiger(len(CARTES_EXISTANTES) == 18 * N_VAL, "Cartes des modèles existants incomplètes.")
apercu = preparer_cartes(CARTES_EXISTANTES)
apercu = (apercu[apercu["correcte"]].groupby(["version", "modele", "graine", "classe"], sort=False)[MESURES_CARTES].mean()
          .groupby(["version", "modele", "classe"], sort=False).mean().reset_index())
print(f"Radios de validation à masque valide : {int(VALIDES_VAL.sum())} sur {N_VAL}.")
print("Décisions correctes, moyenne sur 3 graines : part de la carte dans les poumons, enrichissement pulmonaire "
      "(1 = carte uniforme), concentration pulmonaire (0 = carte uniforme, 1 = tout dans les poumons) et part dans le padding (v2) :")
display(apercu[["version", "modele", "classe", "aire_poumons", "part_poumons", "enrichissement_poumons", "concentration_poumons",
                "enrichissement_poumons_elargis", "part_hors_image"]].round(3))

## 4. Préparer la version v4 et entraîner les six modèles

Chaque radio est recadrée sur la boîte de ses poumons (marge de 10 %), étirée en 224 × 224 puis égalisée. Si le masque est invalide, la radio garde l'image entière : son entrée est alors exactement l'entrée v3 (contrôlé ci-dessous).

Avant l'entraînement, on vérifie ce que les entrées v4 contiennent encore, comme au notebook 12 : l'intensité et la netteté de l'entrée, mais aussi la **géométrie du recadrage** (taille relative de la boîte, proportions, surface des poumons) et le **repli sur l'image entière**. Si l'un d'eux sépare nettement les classes, le recadrage a créé un nouvel indice possible, à signaler dans le mémoire.

L'entraînement reprend exactement le code du notebook 12 : mêmes graines, donc mêmes poids initiaux et même ordre des lots qu'en v2 et v3. Seule l'image d'entrée change. **Cette partie dure plusieurs heures.** Les sauvegardes sont écrites dans `poumons_v4/modeles_v4/<modèle>/graine_<graine>/configuration_<indice>/`.

In [ ]:
def preparer_v4(image, boite):
    """Préparation v4 : boîte des poumons (parts de l'image) étirée en 224 × 224, puis égalisation d'histogramme."""
    gris = image.convert("L")
    largeur, hauteur = gris.size
    x0, y0, x1, y1 = boite
    recadree = gris.resize((TAILLE, TAILLE), Image.Resampling.BILINEAR,
                           box=(x0 * largeur, y0 * hauteur, x1 * largeur, y1 * hauteur))
    return np.array(ImageOps.equalize(recadree), dtype=np.uint8)

def statistiques_entree(tableau_u8):
    """Mêmes statistiques qu'au notebook 12, calculées sur l'entrée du réseau."""
    x = tableau_u8.astype(np.float64) / 255.0
    p05, p50, p95 = np.percentile(x, [5, 50, 95])
    nettete = float(np.abs(np.diff(x, axis=0)).mean() + np.abs(np.diff(x, axis=1)).mean())
    return {"moyenne": float(x.mean()), "ecart_type": float(x.std()), "p05": float(p05), "p50": float(p50),
            "p95": float(p95), "nettete": nettete}

COLONNES_BOITE = ["recadrage_x0", "recadrage_y0", "recadrage_x1", "recadrage_y1"]
IMAGES_V4, SHA_V4, lignes = {}, {}, []
for ensemble, tableau in ENSEMBLES.items():
    infos = QUALITE_INDEX.loc[tableau["chemin_relatif"]]
    boites = infos[COLONNES_BOITE].to_numpy(dtype=np.float64)
    images = np.empty((len(tableau), TAILLE, TAILLE), dtype=np.uint8)
    empreinte = hashlib.sha256()
    for i, ligne in enumerate(tqdm(tableau.to_dict("records"), desc=f"Préparation v4 : {ensemble}")):
        with Image.open(IMAGES / ligne["chemin_relatif"]) as image:
            largeur, hauteur = image.size
            images[i] = preparer_v4(image, tuple(boites[i]))
        empreinte.update(ligne["chemin_relatif"].encode("utf-8") + b"\0")
        empreinte.update(images[i].tobytes())
        x0, y0, x1, y1 = boites[i]
        lignes.append({"ensemble": ensemble, "chemin_relatif": ligne["chemin_relatif"], "groupe": ligne["groupe"],
                       "classe": ligne["classe"], "largeur": largeur, "hauteur": hauteur, "rapport_largeur_hauteur": largeur / hauteur,
                       "repli_image_entiere": float(not bool(infos["masque_valide"].iat[i])),
                       "aire_poumons": float(infos["aire_poumons"].iat[i]),
                       "recadrage_largeur_relative": x1 - x0, "recadrage_hauteur_relative": y1 - y0,
                       "recadrage_rapport": (x1 - x0) * largeur / ((y1 - y0) * hauteur), **statistiques_entree(images[i])})
    IMAGES_V4[ensemble] = images
    SHA_V4[ensemble] = empreinte.hexdigest()
ENTREES_V4 = pd.DataFrame(lignes)
ecrire_csv(ENTREES_V4, SORTIE / "caracteristiques_entrees_v4.csv")

# Contrôle : un masque invalide redonne exactement l'entrée v3.
invalides_val = np.flatnonzero(~VALIDES_VAL)[:5]
for i in invalides_val:
    exiger(np.array_equal(IMAGES_V4["val"][i], EGALISEES[i]), "Repli sur l'image entière : l'entrée v4 devrait être l'entrée v3.")
print(f"Entrées v4 prêtes ({sum(len(v) for v in IMAGES_V4.values())} radios). Repli sur l'image entière contrôlé sur "
      f"{len(invalides_val)} radio(s) de validation à masque invalide.")

def auc_orientee(y, valeurs):
    """AUC de séparation (0,5 = aucune, 1 = parfaite) et classe qui a les valeurs les plus élevées."""
    if np.ptp(valeurs) == 0:
        return 0.5, "aucune"
    auc = roc_auc_score(y, valeurs)
    return max(auc, 1 - auc), ("PNEUMONIA" if auc > 0.5 else ("NORMAL" if auc < 0.5 else "aucune"))

VARIABLES_V4 = {"moyenne": "intensité moyenne de l'entrée v4", "ecart_type": "contraste de l'entrée v4",
                "nettete": "netteté de l'entrée v4", "recadrage_largeur_relative": "largeur de la boîte (part de l'image)",
                "recadrage_hauteur_relative": "hauteur de la boîte (part de l'image)",
                "recadrage_rapport": "proportions de la boîte (largeur/hauteur en pixels)",
                "aire_poumons": "surface des poumons (part de l'image)", "repli_image_entiere": "repli sur l'image entière (masque invalide)",
                "rapport_largeur_hauteur": "format d'origine (métadonnée, rappel)"}
lignes = []
for variable, libelle in VARIABLES_V4.items():
    ligne = {"variable": variable, "description": libelle}
    for ensemble in ["train", "val"]:
        t = ENTREES_V4[ENTREES_V4["ensemble"] == ensemble]
        auc, sens = auc_orientee((t["classe"] == "PNEUMONIA").to_numpy(dtype=int), t[variable].to_numpy(dtype=float))
        ligne[f"auc_separation_{ensemble}"] = auc
        ligne[f"valeurs_plus_elevees_chez_{ensemble}"] = sens
    lignes.append(ligne)
DIAGNOSTIC_V4 = pd.DataFrame(lignes)

def modele_logistique(variables):
    apprentissage = ENTREES_V4[ENTREES_V4["ensemble"] == "train"]
    validation = ENTREES_V4[ENTREES_V4["ensemble"] == "val"]
    modele = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=5000))
    modele.fit(apprentissage[variables].to_numpy(dtype=float), (apprentissage["classe"] == "PNEUMONIA").to_numpy(dtype=int))
    probabilites = modele.predict_proba(validation[variables].to_numpy(dtype=float))[:, 1]
    y = (validation["classe"] == "PNEUMONIA").to_numpy(dtype=int)
    return {"auc_validation": roc_auc_score(y, probabilites),
            "f1_macro_validation": f1_score(y, (probabilites >= 0.5).astype(int), average="macro")}

MODELES_SIMPLES_V4 = pd.DataFrame([
    {"informations_utilisees": "Intensité de l'entrée v4 (moyenne, écart-type, percentiles)",
     **modele_logistique(["moyenne", "ecart_type", "p05", "p50", "p95"])},
    {"informations_utilisees": "Netteté de l'entrée v4", **modele_logistique(["nettete"])},
    {"informations_utilisees": "Géométrie du recadrage (taille et proportions de la boîte, surface des poumons, repli)",
     **modele_logistique(["recadrage_largeur_relative", "recadrage_hauteur_relative", "recadrage_rapport", "aire_poumons",
                          "repli_image_entiere"])},
    {"informations_utilisees": "Format d'origine (métadonnées, rappel du notebook 12)",
     **modele_logistique(["largeur", "hauteur", "rapport_largeur_hauteur"])},
])
ecrire_csv(DIAGNOSTIC_V4, SORTIE / "diagnostic_entrees_v4.csv")
ecrire_csv(MODELES_SIMPLES_V4, SORTIE / "diagnostic_modeles_simples_v4.csv")
print("Ce que les entrées v4 contiennent encore : pouvoir séparateur de chaque variable (AUC : 0,5 = aucun, 1 = parfait)")
display(DIAGNOSTIC_V4.round(3))
print("Petits modèles logistiques entraînés sur l'apprentissage, évalués sur la validation :")
display(MODELES_SIMPLES_V4.round(3))

# Figure : mêmes radios d'apprentissage qu'au notebook 12, en v3 et en v4, avec le contour des poumons sur l'entrée v4.
exemples = pd.concat([train_df[train_df["classe"] == "NORMAL"].head(3), train_df[train_df["classe"] == "PNEUMONIA"].head(3)])
position_train = {chemin: i for i, chemin in enumerate(train_df["chemin_relatif"])}
fig, axes = plt.subplots(2, len(exemples), figsize=(3 * len(exemples), 7.0), constrained_layout=True)
for colonne, ligne in enumerate(exemples.to_dict("records")):
    chemin = ligne["chemin_relatif"]
    with Image.open(IMAGES / chemin) as image:
        largeur, hauteur = image.size
        entree_v3 = egaliser(etirer(image))
    info = QUALITE_INDEX.loc[chemin]
    zone = qualite_masque(*masques_bruts(chemin))[1]
    zone_entree = zone_v4(zone, tuple(float(info[c]) for c in COLONNES_BOITE))
    for rang, (tableau, titre) in enumerate([(entree_v3, f"{ligne['classe']} · {largeur} × {hauteur} px\nv3 : image entière"),
                                             (IMAGES_V4["train"][position_train[chemin]],
                                              "v4 : recadrée sur les poumons" if info["masque_valide"] else "v4 : masque invalide, image entière")]):
        axes[rang, colonne].imshow(tableau, cmap="gray", vmin=0, vmax=255)
        if rang == 1 and info["masque_valide"] and zone_entree.any():
            axes[rang, colonne].contour(zone_entree.astype(float), levels=[0.5], colors=[COULEUR_CONTOUR], linewidths=1.0)
        axes[rang, colonne].set_title(titre, fontsize=9)
        axes[rang, colonne].axis("off")
fig.suptitle("Mêmes radios d'apprentissage : préparation v3 (en haut) et v4 (en bas, contour jaune : poumons)")
enregistrer_figure(fig, SORTIE / "figures" / "preparation_v3_v4.png")
plt.show()
plt.close(fig)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class Entrees(Dataset):
    """Images v4 déjà préparées, gardées en mémoire."""
    def __init__(self, images, etiquettes):
        self.images, self.etiquettes = images, etiquettes
    def __len__(self):
        return len(self.images)
    def __getitem__(self, i):
        x = torch.from_numpy(self.images[i].astype(np.float32) / 255.0).unsqueeze(0)
        return x, int(self.etiquettes[i])

TRAIN_V4, VAL_V4 = Entrees(IMAGES_V4["train"], Y["train"]), Entrees(IMAGES_V4["val"], Y["val"])
effectifs = train_df["classe"].value_counts()
POIDS = torch.tensor([len(train_df) / (2 * effectifs[c]) for c in ["NORMAL", "PNEUMONIA"]], dtype=torch.float32)
with torch.random.fork_rng(devices=[]):
    lot_controle = torch.from_numpy(IMAGES_V4["train"][:LOT].astype(np.float32) / 255.0).unsqueeze(1)
    for nom, (classe, _, nombre_parametres) in ARCHITECTURES.items():
        verification = classe().eval()
        with torch.no_grad():
            scores = verification(lot_controle)
        exiger(tuple(scores.shape) == (LOT, 2) and torch.isfinite(scores).all().item(), f"Sortie inattendue : {nom}.")
        exiger(sum(p.numel() for p in verification.parameters()) == nombre_parametres, f"Nombre de paramètres inattendu : {nom}.")
    del verification, lot_controle
print("Architectures vérifiées ; poids NORMAL / PNEUMONIA :", POIDS.tolist())

def construire_protocole(architecture, graine, lr, dropout):
    return {
        "version": "v4_recadrage_poumons", "architecture": architecture, "graine": graine,
        "lot": LOT, "max_epoques": MAX_EPOQUES, "patience": PATIENCE,
        "critere": "val_f1_macro", "amelioration": "strictement superieure",
        "grille": [[lr, dropout]], "optimiseur": "Adam", "weight_decay": 0.0,
        "pretraitement": PRETRAITEMENTS[V4], "classes": CLASSES, "poids_classes": POIDS.tolist(),
        "sha256_csv": {"train": SHA_CSV["train"], "val": SHA_CSV["val"]},
        "sha256_entrees_v4": {"train": SHA_V4["train"], "val": SHA_V4["val"]},
        "id_protocole_notebook_16": ID_PROTOCOLE, "versions": VERSIONS, "threads_torch": torch.get_num_threads(),
        "limite": "Reglages choisis avec la preparation v2, non reoptimises; recadrage issu d'un segmenteur entraine sur des adultes; test deja consulte (notebook 14).",
    }

def une_epoque(modele, chargeur, critere, optimiseur=None):
    apprentissage = optimiseur is not None
    modele.train(apprentissage)
    pertes, poids_total = 0.0, 0.0
    reels, predits = [], []
    for images, etiquettes in tqdm(chargeur, desc="Apprentissage" if apprentissage else "Validation", leave=False):
        images, etiquettes = images.to(APPAREIL), etiquettes.to(APPAREIL)
        if apprentissage: optimiseur.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(apprentissage):
            scores = modele(images)
            perte = critere(scores, etiquettes)
            if not torch.isfinite(perte).item(): raise RuntimeError("Perte non finie : arrêter et examiner cette configuration.")
            if apprentissage:
                perte.backward()
                optimiseur.step()
        poids = critere.weight[etiquettes].sum().item()
        pertes += perte.item()*poids
        poids_total += poids
        reels.extend(etiquettes.cpu().tolist())
        predits.extend(scores.detach().argmax(1).cpu().tolist())
    cm = confusion_matrix(reels, predits, labels=[0,1])
    tn,fp,fn,tp = cm.ravel()
    mesures = {
        "perte": float(pertes/poids_total), "exactitude": float(accuracy_score(reels,predits)),
        "precision_pneumonie": float(precision_score(reels,predits,zero_division=0)),
        "rappel_pneumonie": float(recall_score(reels,predits,zero_division=0)),
        "specificite_NORMAL": float(tn/(tn+fp)),
        "f1_pneumonie": float(f1_score(reels,predits,zero_division=0)),
        "f1_macro": float(f1_score(reels,predits,labels=[0,1],average="macro",zero_division=0))
    }
    return mesures, cm.tolist()

def entrainer(nom, graine):
    classe, architecture, _ = ARCHITECTURES[nom]
    reglages = CONFIGURATIONS[nom]
    configuration = dict(reglages)
    dossier = dossier_v4(nom, graine)
    dossier.mkdir(parents=True, exist_ok=True)
    protocole = construire_protocole(architecture, graine, reglages["lr"], reglages["dropout"])
    fichier_protocole = dossier / "protocole.json"
    if fichier_protocole.exists():
        exiger(json.loads(fichier_protocole.read_text(encoding="utf-8")) == protocole,
               f"Protocole de reprise différent pour {nom}, graine {graine} : même noyau, mêmes versions et mêmes images nécessaires.")
    else:
        ecrire_json(protocole, fichier_protocole)
    # Même séquence que les notebooks 05 à 12 : poids initiaux et ordre des lots identiques à v2 et v3 pour une même graine.
    torch.manual_seed(graine)
    modele = classe(reglages["dropout"]).to(APPAREIL)
    optimiseur = torch.optim.Adam(modele.parameters(), lr=reglages["lr"])
    critere = nn.CrossEntropyLoss(weight=POIDS.to(APPAREIL))
    generateur = torch.Generator().manual_seed(graine)
    chargeur_train = DataLoader(TRAIN_V4, batch_size=LOT, shuffle=True, generator=generateur, num_workers=0)
    chargeur_val = DataLoader(VAL_V4, batch_size=LOT, shuffle=False, num_workers=0)
    torch.manual_seed(graine)
    historique = []
    def sauvegarder(epoque, matrice=None):
        sauver_torch({"epoque": epoque, "modele": modele.state_dict(), "optimiseur": optimiseur.state_dict(),
                      "historique": historique, "configuration": configuration, "protocole": protocole,
                      "etat_torch": torch.get_rng_state(), "etat_generateur": generateur.get_state(),
                      "matrice_validation": matrice}, dossier / f"epoque_{epoque:03d}.pt")
    disponibles = sorted(dossier.glob("epoque_[0-9][0-9][0-9].pt"))
    if disponibles:
        etat = torch.load(disponibles[-1], map_location="cpu", weights_only=True)
        exiger(etat["configuration"] == configuration and etat["protocole"] == protocole, "Sauvegarde de reprise incompatible.")
        modele.load_state_dict(etat["modele"], strict=True)
        optimiseur.load_state_dict(etat["optimiseur"])
        historique = etat["historique"]
        exiger(etat["epoque"] == len(historique), "Historique de reprise incomplet.")
        torch.set_rng_state(etat["etat_torch"])
        generateur.set_state(etat["etat_generateur"])
        print("Reprise après l'époque", etat["epoque"])
    else:
        sauvegarder(0)
    while len(historique) < MAX_EPOQUES:
        if historique:
            meilleur = max(historique, key=lambda r: r["val_f1_macro"])
            if len(historique) - meilleur["epoque"] >= PATIENCE:
                break
        epoque = len(historique) + 1
        debut = time.perf_counter()
        tr, _ = une_epoque(modele, chargeur_train, critere, optimiseur)
        va, cm = une_epoque(modele, chargeur_val, critere)
        ligne = {"epoque": epoque, "duree_secondes": float(time.perf_counter() - debut)}
        ligne.update({f"train_{k}": v for k, v in tr.items()})
        ligne.update({f"val_{k}": v for k, v in va.items()})
        historique.append(ligne)
        sauvegarder(epoque, cm)
        ecrire_csv(pd.DataFrame(historique), dossier / "historique.csv")
        print(f"Époque {epoque:02d} | F1 macro val {va['f1_macro']:.4f} | rappel {va['rappel_pneumonie']:.4f} | "
              f"spécificité {va['specificite_NORMAL']:.4f} | {ligne['duree_secondes'] / 60:.1f} min")
    meilleur = max(historique, key=lambda r: r["val_f1_macro"])
    source = dossier / f"epoque_{meilleur['epoque']:03d}.pt"
    temporaire = dossier / "meilleur.tmp"
    shutil.copy2(source, temporaire)
    remplacer(temporaire, dossier / "meilleur.pt")
    ecrire_csv(pd.DataFrame(historique), dossier / "historique.csv")
    resume = {**configuration, "epoque_retenue": meilleur["epoque"], "epoque_finale": len(historique),
              "arret": "patience" if len(historique) - meilleur["epoque"] >= PATIENCE else "maximum_epoques",
              "duree_minutes": sum(r["duree_secondes"] for r in historique) / 60,
              "sauvegarde_relative": (dossier / "meilleur.pt").relative_to(PROJET).as_posix(),
              **{k: v for k, v in meilleur.items() if k.startswith("val_")}}
    ecrire_json(resume, dossier / "resume.json")
    return resume

def sauver_torch(objet, chemin):
    temporaire = chemin.with_suffix(".tmp")
    torch.save(objet, temporaire)
    remplacer(temporaire, chemin)

print("Fonctions d'entraînement prêtes. Aucun entraînement lancé jusqu'ici.")

In [ ]:
# Cette cellule dure plusieurs heures. Pour faire une pause : interrompre ; pour reprendre : réexécuter toutes les cellules.
resumes = []
for nom in ARCHITECTURES_NOMS:
    for graine in GRAINES:
        reglages = CONFIGURATIONS[nom]
        print(f"\n===== v4 | {nom} | graine {graine} | LR={reglages['lr']}, Dropout={reglages['dropout']} =====")
        resumes.append({"modele": nom, "graine": graine, **entrainer(nom, graine)})
        RESULTATS_V4 = pd.DataFrame(resumes)
        ecrire_csv(RESULTATS_V4, SORTIE / "resultats_v4.csv")
        display(RESULTATS_V4[["modele", "graine", "epoque_retenue", "epoque_finale", "arret", "val_f1_macro",
                              "val_rappel_pneumonie", "val_specificite_NORMAL"]].round(4))
exiger(len(RESULTATS_V4) == 6, "Certains entraînements sont incomplets.")
print("Les six entraînements v4 sont terminés.")

## 5. Évaluer v4 sur la validation et trancher les prédictions P5 à P8

Les six modèles v4 sont rechargés depuis `meilleur.pt` : leur F1 de validation doit retrouver exactement celui de l'entraînement. Leurs cartes sont ensuite mesurées comme celles des modèles existants.

Comparaisons, toutes appariées par architecture et par graine :

1. **Cartes** : enrichissement pulmonaire des décisions correctes, pour les quatre versions (P5, P6 et leurs analyses de sensibilité).
2. **Format** : corrélation entre le format d'origine et le score, au sein de chaque classe (P7 porte sur les radios PNEUMONIA).
3. **Performances** : F1 macro, AUC, rappel et spécificité (P8 porte sur le F1 macro).
4. **CBAM contre classique** dans chaque version, et attention spatiale du CBAM dans et hors des poumons (descriptif).

In [ ]:
RESULTATS_V4 = pd.read_csv(SORTIE / "resultats_v4.csv")
exiger(len(RESULTATS_V4) == 6 and set(zip(RESULTATS_V4["modele"], RESULTATS_V4["graine"].astype(int)))
       == {(m, g) for m in ARCHITECTURES_NOMS for g in GRAINES}, "resultats_v4.csv doit contenir les six modèles v4.")
MODELES_V4_ORDRE = [(V4, m, g) for m in ARCHITECTURES_NOMS for g in GRAINES]
INFORMATIONS_V4 = []
for ligne in RESULTATS_V4.to_dict("records"):
    nom, graine = ligne["modele"], int(ligne["graine"])
    chemin = dossier_v4(nom, graine) / "meilleur.pt"
    exiger(chemin.is_file(), f"Sauvegarde v4 introuvable : {chemin}")
    modele, epoque = charger_modele(V4, nom, graine, chemin)
    exiger(epoque == int(ligne["epoque_retenue"]), f"Époque v4 incompatible avec resultats_v4.csv : {nom}, graine {graine}.")
    MODELES[(V4, nom, graine)] = modele
    logits = np.concatenate([logits_lot(modele, entrees_val(V4, np.arange(d, min(d + LOT, N_VAL)))) for d in range(0, N_VAL, LOT)])
    f1 = f1_score(REELS, (logits[:, 1] > logits[:, 0]).astype(int), labels=[0, 1], average="macro", zero_division=0)
    exiger(abs(f1 - float(ligne["val_f1_macro"])) < 1e-10, f"F1 recalculé différent pour {nom}, graine {graine}.")
    REFERENCES[(V4, nom, graine)] = logits
    tableau = val_df[["chemin_relatif", "groupe"]].copy()
    tableau["classe_vraie"] = REELS
    tableau["prediction"] = (logits[:, 1] > logits[:, 0]).astype(int)
    tableau["logit_NORMAL"], tableau["logit_PNEUMONIA"] = logits[:, 0], logits[:, 1]
    tableau["modele"], tableau["graine"], tableau["epoque"] = nom, graine, epoque
    ecrire_csv(tableau, SORTIE / "predictions_validation" / f"{nom_fichier(V4, nom, graine)}.csv")
    INFORMATIONS_V4.append({"version": V4, "modele": nom, "graine": graine, "epoque": epoque,
                            "sha256_sauvegarde": empreinte_fichier(chemin), "sauvegarde_relative": chemin.relative_to(PROJET).as_posix()})
print("Modèles v4 rechargés : F1 de validation identiques à ceux de l'entraînement.")
CARTES_V4 = traiter_versions(MODELES_V4_ORDRE)
CARTES_PAR_IMAGE = pd.concat([CARTES_EXISTANTES, CARTES_V4], ignore_index=True)
exiger(len(CARTES_PAR_IMAGE) == 24 * N_VAL, "Cartes incomplètes.")
ecrire_csv(CARTES_PAR_IMAGE, SORTIE / "cartes_par_image.csv")
print("Cartes mesurées pour les 24 modèles.")

In [ ]:
TOUS_MODELES = MODELES_EXISTANTS_ORDRE + MODELES_V4_ORDRE
METRIQUES = ["f1_macro", "auc_roc", "rappel_pneumonie", "specificite_NORMAL", "exactitude"]

def mesures(y, logits):
    prediction = (logits[:, 1] > logits[:, 0]).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    return {"f1_macro": f1_score(y, prediction, labels=[0, 1], average="macro", zero_division=0),
            "auc_roc": roc_auc_score(y, logits[:, 1] - logits[:, 0]),
            "rappel_pneumonie": tp / (tp + fn), "specificite_NORMAL": tn / (tn + fp),
            "exactitude": (tn + tp) / len(y), "faux_positifs": int(fp), "faux_negatifs": int(fn)}

MESURES_VAL = pd.DataFrame([{"version": v, "modele": m, "graine": g, **mesures(REELS, REFERENCES[(v, m, g)])} for (v, m, g) in TOUS_MODELES])
SYNTHESE_VAL = resumer_modeles(MESURES_VAL, ["version", "modele"], METRIQUES + ["faux_positifs", "faux_negatifs"])

lignes = []
for (v, m, g) in TOUS_MODELES:
    score = REFERENCES[(v, m, g)][:, 1] - REFERENCES[(v, m, g)][:, 0]
    for classe in [0, 1]:
        masque = REELS == classe
        lignes.append({"version": v, "modele": m, "graine": g, "classe_vraie": NOMS_CLASSES[classe],
                       "rho_format_score": float(spearmanr(RAPPORTS[masque], score[masque])[0])})
DEPENDANCE = pd.DataFrame(lignes)
DEPENDANCE["abs_rho_format_score"] = DEPENDANCE["rho_format_score"].abs()
RESUME_DEPENDANCE = resumer_modeles(DEPENDANCE, ["classe_vraie", "version", "modele"], ["rho_format_score", "abs_rho_format_score"])

CARTES = preparer_cartes(CARTES_PAR_IMAGE)
PAR_MODELE_CARTES = (CARTES[CARTES["correcte"]].groupby(["version", "modele", "graine", "classe"], sort=False)[MESURES_CARTES]
                     .mean().reset_index())
SYNTHESE_CARTES = resumer_modeles(PAR_MODELE_CARTES, ["version", "modele", "classe"], MESURES_CARTES)
MESURES_ATTENTION = ["attention_poumons", "attention_hors_poumons", "ecart_attention_poumons"]
ATTENTION_PAR_GRAINE = (CARTES[CARTES["modele"] == "CNN_CBAM"].groupby(["version", "graine", "classe"], sort=False)[MESURES_ATTENTION]
                        .mean().reset_index())
RESUME_ATTENTION = resumer_modeles(ATTENTION_PAR_GRAINE, ["version", "classe"], MESURES_ATTENTION)

def pivot_versions(tableau, colonne):
    pivot = tableau.pivot_table(index=["modele", "graine"], columns="version", values=colonne).reset_index()
    pivot.columns.name = None
    for version in VERSIONS_MODELES:
        if version not in pivot.columns:   # version sans aucune décision correcte dans cette classe
            pivot[version] = np.nan
    return pivot

# Tableau apparié v3 / v4 : une ligne par architecture et graine.
pneumonie = PAR_MODELE_CARTES[PAR_MODELE_CARTES["classe"] == "PNEUMONIA"]
normal = PAR_MODELE_CARTES[PAR_MODELE_CARTES["classe"] == "NORMAL"]
ECARTS_V3_V4 = pd.DataFrame({"modele": [m for m in ARCHITECTURES_NOMS for g in GRAINES], "graine": [g for m in ARCHITECTURES_NOMS for g in GRAINES]})
sources = [(MESURES_VAL, m, m) for m in METRIQUES]
sources += [(pneumonie, "enrichissement_poumons", "enrichissement_poumons_PNEUMONIA"),
            (pneumonie, "enrichissement_poumons_elargis", "enrichissement_poumons_elargis_PNEUMONIA"),
            (pneumonie, "concentration_poumons", "concentration_poumons_PNEUMONIA"),
            (pneumonie, "concentration_poumons_elargis", "concentration_poumons_elargis_PNEUMONIA"),
            (pneumonie, "part_poumons", "part_poumons_PNEUMONIA"),
            (normal, "concentration_poumons", "concentration_poumons_NORMAL"),
            (normal, "enrichissement_poumons", "enrichissement_poumons_NORMAL"),
            (DEPENDANCE[DEPENDANCE["classe_vraie"] == "PNEUMONIA"], "abs_rho_format_score", "abs_rho_format_PNEUMONIA"),
            (DEPENDANCE[DEPENDANCE["classe_vraie"] == "NORMAL"], "abs_rho_format_score", "abs_rho_format_NORMAL")]
for tableau, colonne, libelle in sources:
    pivot = pivot_versions(tableau, colonne)
    ECARTS_V3_V4 = ECARTS_V3_V4.merge(pivot[["modele", "graine", V3, V4]].rename(columns={V3: f"{libelle}_v3", V4: f"{libelle}_v4"}),
                                      on=["modele", "graine"], how="left", validate="one_to_one")
    ECARTS_V3_V4[f"{libelle}_ecart_v4_moins_v3"] = ECARTS_V3_V4[f"{libelle}_v4"] - ECARTS_V3_V4[f"{libelle}_v3"]
exiger(len(ECARTS_V3_V4) == 6, "Les six paires v3 / v4 sont nécessaires.")

# Une ligne par modèle v4 : un modèle sans aucune décision PNEUMONIA correcte compte comme cas défavorable (valeur absente).
v4_pneumonie = (pneumonie[pneumonie["version"] == V4].set_index(["modele", "graine"])
                .reindex(pd.MultiIndex.from_tuples([(m, g) for m in ARCHITECTURES_NOMS for g in GRAINES], names=["modele", "graine"])))
for cle, colonne in [("P5", "enrichissement_poumons"), ("P5 sensibilité", "enrichissement_poumons_elargis")]:
    favorables = int((v4_pneumonie[colonne] > 1).sum())
    BILAN[cle] = {"prediction": cle, "enonce": PREDICTIONS[cle], "cas_favorables": favorables, "cas": len(v4_pneumonie),
                  "verdict": verdict(favorables, len(v4_pneumonie))}
for cle, libelle in [("P6", "concentration_poumons_PNEUMONIA"), ("P6 sensibilité", "concentration_poumons_elargis_PNEUMONIA")]:
    favorables = int((ECARTS_V3_V4[f"{libelle}_v4"] > ECARTS_V3_V4[f"{libelle}_v3"]).sum())
    BILAN[cle] = {"prediction": cle, "enonce": PREDICTIONS[cle], "cas_favorables": favorables, "cas": 6, "verdict": verdict(favorables, 6)}
favorables = int((ECARTS_V3_V4["abs_rho_format_PNEUMONIA_v4"] < ECARTS_V3_V4["abs_rho_format_PNEUMONIA_v3"]).sum())
BILAN["P7"] = {"prediction": "P7", "enonce": PREDICTIONS["P7"], "cas_favorables": favorables, "cas": 6, "verdict": verdict(favorables, 6)}
# Arrondi à 12 décimales : une égalité exacte à −0,02 ne doit pas basculer à cause d'un résidu de calcul.
favorables = int((np.round(ECARTS_V3_V4["f1_macro_ecart_v4_moins_v3"].to_numpy(dtype=float), 12) >= -0.02).sum())
BILAN["P8"] = {"prediction": "P8", "enonce": PREDICTIONS["P8"], "cas_favorables": favorables, "cas": 6, "verdict": verdict(favorables, 6)}

# CBAM contre classique dans chaque version (descriptif).
lignes = []
for version in VERSIONS_MODELES:
    for graine in GRAINES:
        ligne = {"version": version, "graine": graine}
        for tableau, colonne, libelle in [(MESURES_VAL, "f1_macro", "f1_macro"), (MESURES_VAL, "auc_roc", "auc_roc"),
                                          (MESURES_VAL, "specificite_NORMAL", "specificite_NORMAL"),
                                          (MESURES_VAL, "rappel_pneumonie", "rappel_pneumonie"),
                                          (pneumonie, "enrichissement_poumons", "enrichissement_poumons_PNEUMONIA"),
                                          (pneumonie, "concentration_poumons", "concentration_poumons_PNEUMONIA"),
                                          (normal, "enrichissement_poumons", "enrichissement_poumons_NORMAL")]:
            valeurs = {m: tableau[(tableau["version"] == version) & (tableau["modele"] == m) & (tableau["graine"] == graine)][colonne]
                       for m in ARCHITECTURES_NOMS}
            ligne[f"{libelle}_CBAM_moins_classique"] = (float(valeurs["CNN_CBAM"].iat[0]) - float(valeurs["CNN_classique"].iat[0])
                                                         if all(len(v) == 1 for v in valeurs.values()) else np.nan)
        lignes.append(ligne)
CBAM_CLASSIQUE = pd.DataFrame(lignes)
colonnes_ecarts = [c for c in CBAM_CLASSIQUE.columns if c.endswith("_CBAM_moins_classique")]
RESUME_CBAM_CLASSIQUE = CBAM_CLASSIQUE.groupby("version", sort=False)[colonnes_ecarts].agg(["mean", "min", "max"])
RESUME_CBAM_CLASSIQUE.columns = [f"{mesure}_{statistique}" for mesure, statistique in RESUME_CBAM_CLASSIQUE.columns]
RESUME_CBAM_CLASSIQUE = RESUME_CBAM_CLASSIQUE.reset_index()
RESUME_CBAM_CLASSIQUE["graines_CBAM_superieur_F1"] = (CBAM_CLASSIQUE.groupby("version", sort=False)["f1_macro_CBAM_moins_classique"]
                                                      .apply(lambda s: int((s > 0).sum())).to_numpy())

ecrire_csv(MESURES_VAL, SORTIE / "mesures_validation.csv")
ecrire_csv(SYNTHESE_VAL, SORTIE / "synthese_validation.csv")
ecrire_csv(RESUME_DEPENDANCE, SORTIE / "resume_dependance_format_validation.csv")
ecrire_csv(PAR_MODELE_CARTES, SORTIE / "cartes_poumons_par_modele.csv")
ecrire_csv(SYNTHESE_CARTES, SORTIE / "synthese_cartes_poumons.csv")
ecrire_csv(RESUME_ATTENTION, SORTIE / "attention_CBAM_poumons.csv")
ecrire_csv(ECARTS_V3_V4, SORTIE / "ecarts_v3_v4_validation.csv")
ecrire_csv(CBAM_CLASSIQUE, SORTIE / "CBAM_classique_validation.csv")
ecrire_csv(RESUME_CBAM_CLASSIQUE, SORTIE / "resume_CBAM_classique_validation.csv")

print("Performances sur la validation (moyenne [min ; max] sur 3 graines) :")
display(SYNTHESE_VAL[["version", "modele", "f1_macro_mean", "f1_macro_min", "f1_macro_max", "auc_roc_mean",
                      "rappel_pneumonie_mean", "specificite_NORMAL_mean"]].round(4))
print("Décisions correctes : part de la carte dans les poumons, enrichissement (1 = uniforme) et concentration (0 = uniforme) pulmonaires, moyenne sur 3 graines :")
display(SYNTHESE_CARTES[["version", "modele", "classe", "aire_poumons_mean", "part_poumons_mean", "enrichissement_poumons_mean",
                         "enrichissement_poumons_min", "enrichissement_poumons_max", "concentration_poumons_mean",
                         "concentration_poumons_min", "concentration_poumons_max", "enrichissement_poumons_elargis_mean",
                         "part_hors_image_mean", "carte_decision_nulle_mean"]].round(3))
print("Écarts v4 − v3 appariés par architecture et graine :")
display(ECARTS_V3_V4[["modele", "graine", "f1_macro_ecart_v4_moins_v3", "specificite_NORMAL_ecart_v4_moins_v3",
                      "auc_roc_ecart_v4_moins_v3", "enrichissement_poumons_PNEUMONIA_ecart_v4_moins_v3",
                      "concentration_poumons_PNEUMONIA_ecart_v4_moins_v3",
                      "abs_rho_format_PNEUMONIA_ecart_v4_moins_v3"]].round(4))
print("Dépendance au format (ρ de Spearman entre format d'origine et score, au sein de chaque classe) :")
display(RESUME_DEPENDANCE[["classe_vraie", "version", "modele", "rho_format_score_mean", "rho_format_score_min",
                           "rho_format_score_max"]].round(3))
print("CBAM − classique, apparié par graine (descriptif) :")
display(RESUME_CBAM_CLASSIQUE.round(4))
print("Attention spatiale du CBAM : valeur moyenne dans les poumons et dans le reste de l'image (0 à 1) :")
display(RESUME_ATTENTION.round(4))
print("Prédictions P5 à P8 :")
display(pd.DataFrame([BILAN[p] for p in ORDRE_BILAN])[["prediction", "cas_favorables", "cas", "verdict"]])

# Figure 1 : enrichissement (lecture intuitive) puis concentration (utilisée par P6), quatre versions.
from matplotlib.lines import Line2D
for cle_figure, mesure_a, mesure_b, nom_mesure, repere, fichier_figure in [
        ("enrichissement", "enrichissement_poumons", "enrichissement_poumons_elargis", "Enrichissement pulmonaire\n(1 = carte uniforme)", 1,
         "poumons_enrichissement_versions.png"),
        ("concentration", "concentration_poumons", "concentration_poumons_elargis",
         "Concentration pulmonaire\n(0 = carte uniforme, 1 = tout dans les poumons)", 0, "poumons_concentration_versions.png")]:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9.8))
    for rang, (colonne, libelle) in enumerate([(mesure_a, "zone pulmonaire"),
                                                (mesure_b, f"zone pulmonaire élargie de {DILATATION} pixels")]):
        for position, classe in enumerate(["NORMAL", "PNEUMONIA"]):
            axe = axes[rang, position]
            points_par_graine(axe, PAR_MODELE_CARTES[PAR_MODELE_CARTES["classe"] == classe], colonne, VERSIONS_MODELES)
            axe.axhline(repere, color=ENCRE_DISCRETE, linewidth=1, linestyle="--")
            axe.set_title(f"Décisions correctes sur les radios {classe}\n{libelle}", fontsize=11)
            axe.set_ylabel(nom_mesure)
    fig.suptitle(f"{cle_figure.capitalize()} des cartes Grad-CAM dans les poumons, validation (points : graines ; trait : moyenne)")
    fig.tight_layout()
    # Légende construite pour les quatre versions, même si une version n'a aucun point dans le premier panneau.
    poignees = [Line2D([], [], marker="o", linestyle="", markersize=8, markerfacecolor=COULEURS[v], markeredgecolor="white", label=v)
                for v in VERSIONS_MODELES]
    fig.legend(handles=poignees, loc="upper center", ncol=4, frameon=False, fontsize=10, bbox_to_anchor=(0.5, 0.0))
    enregistrer_figure(fig, SORTIE / "figures" / fichier_figure)
    plt.show()
    plt.close(fig)

# Figure 2 : performances de validation, v3 et v4.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for axe, (colonne, titre) in zip(axes, [("f1_macro", "F1 macro"), ("specificite_NORMAL", "Spécificité NORMAL"), ("auc_roc", "AUC")]):
    points_par_graine(axe, MESURES_VAL, colonne, [V3, V4])
    axe.set_title(titre, fontsize=11)
axes[0].legend(fontsize=9, frameon=False, loc="best")
fig.suptitle("Validation : v3 et v4, à architecture et graine égales (points : graines ; trait : moyenne)")
fig.tight_layout()
enregistrer_figure(fig, SORTIE / "figures" / "performances_validation_v3_v4.png")
plt.show()
plt.close(fig)

# Figure 3 : cartes moyennes des décisions correctes, v3 et v4, avec le contour moyen des poumons.
def carte_moyenne(version, nom, type_carte, classe):
    cartes = []
    for graine in GRAINES:
        with np.load(SORTIE / "cartes" / f"{nom_fichier(version, nom, graine)}.npz") as donnees:
            cle = f"{type_carte}__{classe}"
            if f"{cle}__somme" in donnees.files and int(donnees[f"{cle}__nombre"]) > 0:
                cartes.append(donnees[f"{cle}__somme"].astype(np.float64) / int(donnees[f"{cle}__nombre"]))
    return np.mean(cartes, axis=0) if cartes else None

fonds = {V3: EGALISEES, V4: IMAGES_V4["val"]}
zones_moyennes = {V3: ZONES_V3, V4: ZONES_V4}
fig, axes = plt.subplots(2, 4, figsize=(15, 8.6), constrained_layout=True)
for ligne_figure, classe in enumerate([0, 1]):
    selection = VALIDES_VAL & (REELS == classe)
    for colonne, (version, nom) in enumerate([(v, m) for v in [V3, V4] for m in ARCHITECTURES_NOMS]):
        axe = axes[ligne_figure, colonne]
        axe.imshow(fonds[version][selection].mean(axis=0) / 255.0, cmap="gray", vmin=0, vmax=1)
        carte = carte_moyenne(version, nom, "decision", classe)
        if carte is not None:
            axe.imshow(carte / carte.max(), cmap="jet", alpha=0.45, vmin=0, vmax=1)
        zone = zones_moyennes[version][selection].mean(axis=0)
        if zone.max() > 0.5:
            axe.contour(zone, levels=[0.5], colors=[COULEUR_CONTOUR], linewidths=1.4)
        axe.set_title(f"{LIBELLES_MODELES[nom]}, {version}\nradios {NOMS_CLASSES[classe]} (décisions correctes)", fontsize=9)
        axe.axis("off")
fig.suptitle("Cartes Grad-CAM moyennes sur la validation (3 graines ; masques valides)\n"
             "Fond : entrée moyenne ; rouge : zones les plus utilisées ; contour jaune : zone où les poumons sont présents sur au moins la moitié des radios")
enregistrer_figure(fig, SORTIE / "figures" / "cartes_moyennes_v3_v4.png")
plt.show()
plt.close(fig)

In [ ]:
def afficher(axe, image, carte=None, attention=False):
    axe.imshow(image, cmap="gray", vmin=0, vmax=1)
    if carte is not None:
        if attention:
            axe.imshow(carte, cmap="viridis", alpha=0.55, vmin=float(carte.min()), vmax=float(carte.max()))
        else:
            axe.imshow(carte, cmap="jet", alpha=0.45, vmin=0, vmax=1)
    axe.set_xticks([])
    axe.set_yticks([])

def cartes_exemples(version, nom, indices, type_carte):
    """Cartes 224 × 224 moyennées sur les trois graines, et nombre de graines qui prédisent PNEUMONIA."""
    x = entrees_val(version, indices)
    cumul = np.zeros((len(indices), TAILLE, TAILLE))
    votes = np.zeros(len(indices), dtype=int)
    for graine in GRAINES:
        lg, signee, spatiale = calculer_lot(nom, MODELES[(version, nom, graine)], x)
        pneumonie_predite = lg[:, 1] > lg[:, 0]
        votes += pneumonie_predite.astype(int)
        if type_carte == "attention":
            cumul += agrandir(spatiale)
            continue
        choisies = np.where(pneumonie_predite[:, None, None], np.maximum(signee, 0.0), np.maximum(-signee, 0.0))
        cartes = agrandir(choisies)
        maxima = cartes.reshape(len(indices), -1).max(axis=1)[:, None, None]
        cumul += np.divide(cartes, maxima, out=np.zeros_like(cartes), where=maxima > 0)
    return cumul / len(GRAINES), votes

# Exemples fixés à l'avance, comme aux notebooks 11 et 13 : les trois premières radios NORMAL et PNEUMONIA de la validation.
exemples = np.array(list(np.flatnonzero(REELS == 0)[:3]) + list(np.flatnonzero(REELS == 1)[:3]))
COLONNES_EXEMPLES = [
    (V3, None, "image", "Entrée v3\n(image entière)"),
    (V3, "CNN_classique", "decision", "Classique v3\nGrad-CAM décision"),
    (V3, "CNN_CBAM", "decision", "CBAM v3\nGrad-CAM décision"),
    (V4, None, "image", "Entrée v4\n(recadrée sur les poumons)"),
    (V4, "CNN_classique", "decision", "Classique v4\nGrad-CAM décision"),
    (V4, "CNN_CBAM", "decision", "CBAM v4\nGrad-CAM décision"),
    (V4, "CNN_CBAM", "attention", "CBAM v4\nattention spatiale"),
]
resultats_exemples = {(version, nom, type_carte): cartes_exemples(version, nom, exemples, type_carte)
                      for version, nom, type_carte, _ in COLONNES_EXEMPLES if nom is not None}
fig, axes = plt.subplots(len(exemples), len(COLONNES_EXEMPLES), figsize=(2.4 * len(COLONNES_EXEMPLES), 2.55 * len(exemples)))
for ligne, i in enumerate(exemples):
    for colonne, (version, nom, type_carte, titre) in enumerate(COLONNES_EXEMPLES):
        axe = axes[ligne, colonne]
        image = entrees_val(version, [i])[0, 0]
        if nom is None:
            afficher(axe, image)
        else:
            cartes, votes = resultats_exemples[(version, nom, type_carte)]
            afficher(axe, image, cartes[ligne], attention=(type_carte == "attention"))
            if type_carte == "decision":
                axe.text(0.03, 0.03, f"PNEUMONIA : {votes[ligne]}/3", transform=axe.transAxes, fontsize=7, color="white",
                         bbox={"facecolor": "black", "alpha": 0.6, "pad": 1.5, "edgecolor": "none"})
        zone = geometrie(version, i)[0]
        if VALIDES_VAL[i] and zone.any():
            axe.contour(zone.astype(float), levels=[0.5], colors=[COULEUR_CONTOUR], linewidths=0.9)
        if ligne == 0:
            axe.set_title(titre, fontsize=8)
    axes[ligne, 0].set_ylabel(f"{NOMS_CLASSES[int(REELS[i])]}\n{Path(IDENTIFIANTS[i]).name[:22]}", fontsize=8)
fig.suptitle("Exemples fixés à l'avance (trois premières radios NORMAL et PNEUMONIA de la validation) : v3 et v4\n"
             "Cartes moyennées sur 3 graines ; contour jaune : poumons ; en bas à gauche : nombre de graines qui prédisent PNEUMONIA")
fig.tight_layout()
enregistrer_figure(fig, SORTIE / "figures" / "exemples_v3_v4.png")
plt.show()
plt.close(fig)

## 6. Test : évaluation exploratoire de v4

**Rien dans cette section ne change les verdicts H1 à H4 du notebook 14.** Le test y a déjà été utilisé, et l'idée de v4 vient en partie de ses résultats (spécificité faible, radios NORMAL plus larges dans le test). Les modèles v4 sont évalués ici pour décrire l'effet du recadrage, sans verdict.

Même méthode qu'au notebook 14 : scores des six modèles v4 sur les 618 radios, puis rééchantillonnage par patient (2 000 tirages, même graine). Les tirages sont exactement ceux du notebook 14 : les scores v3 du notebook 14 doivent retrouver sa distribution enregistrée (contrôle automatique). Comparaisons décrites avec leur intervalle à 95 % : v4 − v3 pour chaque architecture et pour les deux, et CBAM − classique en v3 et en v4. On regarde aussi les faux positifs selon le format d'origine, là où le raccourci faisait le plus de dégâts.

In [ ]:
VALIDES_TEST = QUALITE_INDEX.loc[IDENTIFIANTS_TEST, "masque_valide"].to_numpy(dtype=bool)
x_test_v4 = (IMAGES_V4["test"].astype(np.float32) / 255.0)[:, None]
MODELES_TEST = [(v, m, g) for v in [V3, V4] for m in ARCHITECTURES_NOMS for g in GRAINES]
SCORES_TEST = {}
for version, nom, graine in MODELES_TEST:
    if version == V3:
        SCORES_TEST[(version, nom, graine)] = PREDICTIONS_TEST_V3[(nom, graine)]["score_pneumonie"].to_numpy(dtype=np.float64)
        continue
    fichier = SORTIE / "predictions_test" / f"{nom_fichier(version, nom, graine)}.csv"
    if fichier.exists():
        tableau = pd.read_csv(fichier, dtype={"groupe": str}, float_precision="round_trip")
        exiger(len(tableau) == N_TEST and (tableau["id_protocole"] == ID_PROTOCOLE).all()
               and tableau["chemin_relatif"].tolist() == IDENTIFIANTS_TEST, f"{fichier.name} : résultats existants incompatibles.")
    else:
        logits = np.concatenate([logits_lot(MODELES[(version, nom, graine)], x_test_v4[d:d + LOT]) for d in range(0, N_TEST, LOT)])
        exiger(logits.shape == (N_TEST, 2) and bool(np.isfinite(logits).all()), f"Scores du test invalides : {nom}, graine {graine}.")
        tableau = test_df[["chemin_relatif", "groupe"]].copy()
        tableau.insert(0, "graine", graine)
        tableau.insert(0, "modele", nom)
        tableau.insert(0, "version", version)
        tableau["classe_vraie"] = Y_TEST
        tableau["rapport_largeur_hauteur"] = RAPPORTS_TEST
        tableau["masque_valide"] = VALIDES_TEST
        tableau["logit_NORMAL"], tableau["logit_PNEUMONIA"] = logits[:, 0], logits[:, 1]
        tableau["score_pneumonie"] = logits[:, 1] - logits[:, 0]
        tableau["prediction"] = (logits[:, 1] > logits[:, 0]).astype(int)
        tableau["statut"] = "exploratoire : test deja utilise au notebook 14"
        tableau["id_protocole"] = ID_PROTOCOLE
        ecrire_csv(tableau, fichier)
    SCORES_TEST[(version, nom, graine)] = tableau["score_pneumonie"].to_numpy(dtype=np.float64)

def mesures_test(y, score, rapports):
    prediction = (score > 0).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    return {"f1_macro": f1_score(y, prediction, labels=[0, 1], average="macro", zero_division=0),
            "auc_roc": roc_auc_score(y, score), "rappel_pneumonie": tp / (tp + fn), "specificite_NORMAL": tn / (tn + fp),
            "precision_pneumonie": tp / (tp + fp) if tp + fp > 0 else 0.0, "exactitude": (tn + tp) / len(y),
            "taux_faux_positifs_pct": 100 * fp / (tn + fp), "faux_positifs": int(fp), "faux_negatifs": int(fn),
            "rho_format_score_NORMAL": float(spearmanr(rapports[y == 0], score[y == 0])[0])}

MESURES_TEST = pd.DataFrame([{"version": v, "modele": m, "graine": g, **mesures_test(Y_TEST, SCORES_TEST[(v, m, g)], RAPPORTS_TEST)}
                             for (v, m, g) in MODELES_TEST])
# Contrôle : les mesures v3 retrouvent celles du notebook 14.
mesures_14 = pd.read_csv(DOSSIER_14 / "mesures_test_par_modele.csv")
for (v, m, g) in MODELES_TEST[:6]:
    attendu = mesures_14[(mesures_14["version"] == v) & (mesures_14["modele"] == m) & (mesures_14["graine"] == g)].iloc[0]
    calcule = MESURES_TEST[(MESURES_TEST["version"] == v) & (MESURES_TEST["modele"] == m) & (MESURES_TEST["graine"] == g)].iloc[0]
    for mesure in ["f1_macro", "auc_roc", "specificite_NORMAL", "rappel_pneumonie"]:
        exiger(abs(float(attendu[mesure]) - float(calcule[mesure])) < 1e-9, f"Mesure v3 différente du notebook 14 : {mesure}, {m}, graine {g}.")

METRIQUES_BOOT = ["f1_macro", "auc_roc", "rappel_pneumonie", "specificite_NORMAL", "taux_faux_positifs_pct", "rho_format_score_NORMAL"]
INDEX = {cle: k for k, cle in enumerate(MODELES_TEST)}
SCORES = np.stack([SCORES_TEST[cle] for cle in MODELES_TEST])

def mesures_rapides(y, score, rangs_rapport_normal):
    """Mêmes mesures que mesures_test, en version rapide pour le rééchantillonnage (notebook 14)."""
    positif, predit = y == 1, score > 0
    tp, fn = int(np.sum(predit & positif)), int(np.sum(~predit & positif))
    fp, tn = int(np.sum(predit & ~positif)), int(np.sum(~predit & ~positif))
    f1_pneumonie = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn > 0 else 0.0
    f1_normal = 2 * tn / (2 * tn + fn + fp) if 2 * tn + fn + fp > 0 else 0.0
    n1, n0 = tp + fn, tn + fp
    rangs = rankdata(score)
    auc = (rangs[positif].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
    rho = np.corrcoef(rangs_rapport_normal, rankdata(score[~positif]))[0, 1]
    return (0.5 * (f1_pneumonie + f1_normal), auc, tp / n1, tn / n0, 100 * fp / n0, rho)

rangs_complets = rankdata(RAPPORTS_TEST[Y_TEST == 0])
for cle in MODELES_TEST:
    for nom_mesure, valeur in zip(METRIQUES_BOOT, mesures_rapides(Y_TEST, SCORES[INDEX[cle]], rangs_complets)):
        exiger(abs(valeur - float(MESURES_TEST.iloc[INDEX[cle]][nom_mesure])) < 1e-9, f"Mesure rapide incorrecte : {nom_mesure}, {cle}.")

# Rééchantillonnage par patient : même algorithme et même graine qu'au notebook 14, donc mêmes tirages.
groupes = test_df["groupe"].to_numpy()
uniques = pd.unique(groupes)
indices_groupe = {g: np.flatnonzero(groupes == g) for g in uniques}
par_classe = {c: [indices_groupe[g] for g in uniques if Y_TEST[indices_groupe[g][0]] == c] for c in [0, 1]}
generateur = np.random.Generator(np.random.PCG64(GRAINE_BOOTSTRAP))
BOOT = np.empty((N_BOOTSTRAP, len(MODELES_TEST), len(METRIQUES_BOOT)), dtype=np.float64)
for b in tqdm(range(N_BOOTSTRAP), desc="Rééchantillonnage par patient"):
    morceaux = []
    for c in [0, 1]:
        tires = generateur.integers(0, len(par_classe[c]), size=len(par_classe[c]))
        morceaux.extend(par_classe[c][k] for k in tires)
    idx = np.concatenate(morceaux)
    y = Y_TEST[idx]
    rangs_rapport = rankdata(RAPPORTS_TEST[idx][y == 0])
    for k in range(len(MODELES_TEST)):
        BOOT[b, k] = mesures_rapides(y, SCORES[k, idx], rangs_rapport)
exiger(np.isfinite(BOOT).all(), "Valeurs non finies dans le rééchantillonnage.")
with np.load(DOSSIER_14 / "bootstrap.npz") as donnees:
    noms_14, distribution_14 = [str(n) for n in donnees["modeles"]], donnees["distribution"]
    exiger([str(m) for m in donnees["metriques"]] == METRIQUES_BOOT, "Mesures du rééchantillonnage différentes du notebook 14.")
for cle in MODELES_TEST[:6]:
    k14 = noms_14.index("|".join(map(str, cle)))
    exiger(np.allclose(BOOT[:, INDEX[cle]], distribution_14[:, k14], rtol=0.0, atol=1e-9),
           "Les tirages diffèrent de ceux du notebook 14 : comparaison impossible.")
print("Rééchantillonnage : tirages identiques à ceux du notebook 14 (contrôlé sur les six modèles v3).")

def estimer(mesure, cas):
    """cas : liste de paires (a, b) pour une différence a − b, ou de modèles seuls (moyenne)."""
    k = METRIQUES_BOOT.index(mesure)
    valeur = lambda cle: float(MESURES_TEST.iloc[INDEX[cle]][mesure])
    if isinstance(cas[0][0], tuple):
        valeurs = np.array([valeur(a) - valeur(b) for a, b in cas])
        distribution = np.mean([BOOT[:, INDEX[a], k] - BOOT[:, INDEX[b], k] for a, b in cas], axis=0)
    else:
        valeurs = np.array([valeur(a) for a in cas])
        distribution = np.mean([BOOT[:, INDEX[a], k] for a in cas], axis=0)
    valeurs, distribution = np.round(valeurs, 12), np.round(distribution, 12)
    bas, haut = np.round(np.percentile(distribution, [2.5, 97.5]), 12)
    return valeurs, float(np.round(valeurs.mean(), 12)), float(bas), float(haut)

MESURES_COMPAREES = [("f1_macro", 1.0, "F1 macro"), ("specificite_NORMAL", 100.0, "points de %"),
                     ("rappel_pneumonie", 100.0, "points de %"), ("auc_roc", 1.0, "AUC")]
lignes = []
for mesure, echelle, unite in MESURES_COMPAREES:
    groupes_cas = [(f"v4 − v3, {LIBELLES_MODELES[nom]}", [((V4, nom, g), (V3, nom, g)) for g in GRAINES]) for nom in ARCHITECTURES_NOMS]
    groupes_cas.append(("v4 − v3, les deux architectures", [((V4, nom, g), (V3, nom, g)) for nom in ARCHITECTURES_NOMS for g in GRAINES]))
    groupes_cas += [(f"CBAM − classique, {version}", [((version, "CNN_CBAM", g), (version, "CNN_classique", g)) for g in GRAINES])
                    for version in [V3, V4]]
    for libelle, cas in groupes_cas:
        valeurs, estimation, bas, haut = estimer(mesure, cas)
        signe = np.sign(estimation)
        lignes.append({"comparaison": libelle, "mesure": mesure, "unite": unite, "estimation": estimation * echelle,
                       "ic95_bas": bas * echelle, "ic95_haut": haut * echelle,
                       "intervalle_exclut_0": bool(bas > 0 or haut < 0),
                       "cas_dans_le_sens_de_l_estimation": int(np.sum(np.sign(valeurs) == signe)) if signe != 0 else 0,
                       "cas": len(valeurs), "valeurs_par_cas": " ; ".join(f"{v * echelle:+.4f}" for v in valeurs)})
COMPARAISONS_TEST = pd.DataFrame(lignes)
lignes = []
for version in [V3, V4]:
    for nom in ARCHITECTURES_NOMS:
        for mesure in METRIQUES_BOOT:
            valeurs, estimation, bas, haut = estimer(mesure, [(version, nom, g) for g in GRAINES])
            lignes.append({"version": version, "modele": nom, "mesure": mesure, "moyenne_3_graines": estimation,
                           "ic95_bas": bas, "ic95_haut": haut, "minimum_graines": valeurs.min(), "maximum_graines": valeurs.max()})
INTERVALLES_TEST = pd.DataFrame(lignes)

LIBELLES_TIERS = ["format le plus étroit", "format intermédiaire", "format le plus large"]
tiers = {}
for classe in [0, 1]:
    bornes = np.quantile(RAPPORTS_TEST[Y_TEST == classe], [1 / 3, 2 / 3])
    tiers[classe] = np.digitize(RAPPORTS_TEST, bornes)
lignes = []
for cle in MODELES_TEST:
    prediction = (SCORES_TEST[cle] > 0).astype(int)
    for classe, erreur in [(0, "faux positifs (radios NORMAL)"), (1, "faux négatifs (radios PNEUMONIA)")]:
        for rang, libelle in enumerate(LIBELLES_TIERS):
            masque = (Y_TEST == classe) & (tiers[classe] == rang)
            lignes.append({"version": cle[0], "modele": cle[1], "graine": cle[2], "erreur": erreur, "tiers_format": libelle,
                           "n_images": int(masque.sum()), "taux_erreur_pct": 100 * float((prediction[masque] != classe).mean())})
ERREURS_FORMAT_TEST = pd.DataFrame(lignes)
RESUME_ERREURS_FORMAT_TEST = (ERREURS_FORMAT_TEST.groupby(["erreur", "version", "modele", "tiers_format"], sort=False)["taux_erreur_pct"]
                              .mean().unstack("tiers_format")[LIBELLES_TIERS].reset_index())

ecrire_csv(MESURES_TEST, SORTIE / "test_exploratoire_mesures.csv")
ecrire_csv(COMPARAISONS_TEST, SORTIE / "test_exploratoire_comparaisons.csv")
ecrire_csv(INTERVALLES_TEST, SORTIE / "test_exploratoire_intervalles.csv")
ecrire_csv(RESUME_ERREURS_FORMAT_TEST, SORTIE / "test_exploratoire_erreurs_format.csv")
print(f"Test (exploratoire) : masques valides pour {100 * VALIDES_TEST.mean():.1f} % des radios.")
print("Performances sur le test, moyenne sur 3 graines [IC 95 %] :")
apercu = INTERVALLES_TEST[INTERVALLES_TEST["mesure"].isin(["f1_macro", "auc_roc", "rappel_pneumonie", "specificite_NORMAL"])]
display(apercu.round(4))
print("Comparaisons exploratoires (sans verdict) :")
display(COMPARAISONS_TEST[["comparaison", "mesure", "unite", "estimation", "ic95_bas", "ic95_haut",
                           "cas_dans_le_sens_de_l_estimation", "cas"]].round(4))
print("Taux d'erreur (%) par tiers de format sur le test, moyenne sur 3 graines :")
display(RESUME_ERREURS_FORMAT_TEST.round(1))

fig, axes = plt.subplots(2, 2, figsize=(14, 9.5))
panneaux = [("f1_macro", "F1 macro"), ("auc_roc", "AUC"), ("rappel_pneumonie", "Rappel PNEUMONIA"), ("specificite_NORMAL", "Spécificité NORMAL")]
decalages = {V3: -0.12, V4: 0.12}
for axe, (mesure, titre) in zip(axes.ravel(), panneaux):
    for version in [V3, V4]:
        for nom, x in POSITIONS_MODELES.items():
            position = x + decalages[version]
            y = MESURES_TEST[(MESURES_TEST["version"] == version) & (MESURES_TEST["modele"] == nom)][mesure].to_numpy(dtype=float)
            ic = INTERVALLES_TEST[(INTERVALLES_TEST["version"] == version) & (INTERVALLES_TEST["modele"] == nom)
                                  & (INTERVALLES_TEST["mesure"] == mesure)].iloc[0]
            axe.vlines(position, ic["ic95_bas"], ic["ic95_haut"], color=COULEURS[version], linewidth=1.2, alpha=0.7, zorder=1)
            axe.scatter(np.full(len(y), position), y, s=64, color=COULEURS[version], edgecolors="white", linewidths=1.5,
                        zorder=3, label=version if nom == "CNN_classique" else None)
            axe.hlines(y.mean(), position - 0.07, position + 0.07, color=COULEURS[version], linewidth=2, zorder=2)
    axe.set_xticks(list(POSITIONS_MODELES.values()), [LIBELLES_MODELES[m] for m in POSITIONS_MODELES])
    axe.set_xlim(-0.5, 1.5)
    axe.set_title(titre, fontsize=11)
    styliser(axe)
axes[0, 0].legend(fontsize=9, frameon=False, loc="best")
fig.suptitle("Test, analyse exploratoire : v3 (notebook 14) et v4 (points : graines ; trait : moyenne ; barre : IC 95 % de la moyenne)")
fig.tight_layout()
enregistrer_figure(fig, SORTIE / "figures" / "test_exploratoire_v3_v4.png")
plt.show()
plt.close(fig)

In [ ]:
exiger(set(BILAN) == set(ORDRE_BILAN), "Exécuter toutes les cellules dans l'ordre pour obtenir le bilan complet des prédictions.")
BILAN_PREDICTIONS = pd.DataFrame([BILAN[p] for p in ORDRE_BILAN])
BILAN_PREDICTIONS["regle"] = REGLE
BILAN_PREDICTIONS["resultats_v4_fragiles"] = FRAGILE
ecrire_csv(BILAN_PREDICTIONS, SORTIE / "bilan_predictions.csv")
print("Bilan des prédictions écrites avant l'analyse :")
display(BILAN_PREDICTIONS[["prediction", "cas_favorables", "cas", "verdict", "resultats_v4_fragiles"]])

def estimation_test(comparaison, mesure):
    ligne = COMPARAISONS_TEST[(COMPARAISONS_TEST["comparaison"] == comparaison) & (COMPARAISONS_TEST["mesure"] == mesure)].iloc[0]
    return {"estimation": float(ligne["estimation"]), "ic95": [float(ligne["ic95_bas"]), float(ligne["ic95_haut"])]}

ecrire_json({"id_protocole": ID_PROTOCOLE,
             "masques_valides_pct": {e: float(100 * QUALITE[QUALITE["ensemble"] == e]["masque_valide"].mean()) for e in ENSEMBLES},
             "resultats_v4_fragiles": FRAGILE, "sens_lateralite": float(SENS_LATERALITE), "modeles_v4": INFORMATIONS_V4,
             "bilan": [{"prediction": p, "cas_favorables": int(BILAN[p]["cas_favorables"]), "verdict": BILAN[p]["verdict"]} for p in ORDRE_BILAN],
             "test_exploratoire": {"v4_moins_v3_f1_macro": estimation_test("v4 − v3, les deux architectures", "f1_macro"),
                                   "v4_moins_v3_specificite_points": estimation_test("v4 − v3, les deux architectures", "specificite_NORMAL"),
                                   "CBAM_moins_classique_v4_f1_macro": estimation_test(f"CBAM − classique, {V4}", "f1_macro")}},
            SORTIE / "termine.json")

# Un seul fichier à envoyer : les tableaux utiles et toutes les figures, réunis dans une archive zip.
FICHIERS_A_ENVOYER = ["bilan_predictions.csv", "resume_qualite_masques.csv", "raisons_masques_invalides.csv", "qualite_masques.csv",
                      "diagnostic_entrees_v4.csv", "diagnostic_modeles_simples_v4.csv", "resultats_v4.csv", "synthese_validation.csv",
                      "ecarts_v3_v4_validation.csv", "resume_dependance_format_validation.csv", "synthese_cartes_poumons.csv",
                      "cartes_poumons_par_modele.csv", "attention_CBAM_poumons.csv", "resume_CBAM_classique_validation.csv",
                      "test_exploratoire_mesures.csv", "test_exploratoire_comparaisons.csv", "test_exploratoire_intervalles.csv",
                      "test_exploratoire_erreurs_format.csv", "termine.json"]
archive = SORTIE / "a_envoyer_notebook_16.zip"
temporaire = archive.with_suffix(".tmp")
with zipfile.ZipFile(temporaire, "w", compression=zipfile.ZIP_DEFLATED) as fichier_zip:
    for nom_tableau in FICHIERS_A_ENVOYER:
        fichier_zip.write(SORTIE / nom_tableau, arcname=nom_tableau)
    for figure in sorted((SORTIE / "figures").glob("*.png")):
        fichier_zip.write(figure, arcname=f"figures/{figure.name}")
remplacer(temporaire, archive)
print("Terminé. Pour transmettre les résultats, un seul fichier suffit (tableaux et figures) :")
print(" ", archive)

## Lecture des résultats

**Qualité des masques (à regarder en premier).** Si le notebook s'arrête avec le message « Arrêt de sécurité », c'est que moins de 50 % des masques sont valides dans une classe : les figures et les CSV de qualité sont déjà écrits, me les transmettre. Ouvrir les deux figures `masques_*.png`. Les contours jaunes doivent suivre les champs pulmonaires. Un segmenteur entraîné sur des adultes peut se tromper chez l'enfant : poumon manquant, contour qui déborde sur l'abdomen, opacité dense laissée hors du masque. Le tableau `resume_qualite_masques.csv` donne la part de masques valides par ensemble et par classe. **Une différence nette entre NORMAL et PNEUMONIA** (par exemple plus d'échecs chez les PNEUMONIA) est à signaler : le recadrage ne serait alors pas le même pour les deux classes.

**Cartes des modèles existants (descriptif).** Un enrichissement pulmonaire inférieur à 1 signifie que la carte évite les poumons : le modèle décide surtout avec autre chose (bords, padding, marqueurs). C'est la mesure qui remplace le « centre » des notebooks 11 et 13 dans le mémoire.

**P5 et P6 : la v4 regarde-t-elle les poumons ?** P5 confirmée : dans l'image recadrée, les cartes des décisions PNEUMONIA sont plus concentrées dans les poumons qu'une carte uniforme (enrichissement > 1, ce qui équivaut à une concentration > 0). P6 confirmée : l'indice de concentration (0 = carte uniforme, 1 = toute la carte dans les poumons) est plus élevé qu'en v3. L'enrichissement, plus intuitif, n'est pas utilisé pour comparer v3 et v4 : son maximum est plus bas en v4, où les poumons remplissent presque toute l'entrée. Comparer avec les analyses de sensibilité (zone élargie). Si P5 et P6 ne tiennent qu'avec la zone élargie, les cartes se placent au bord des poumons, là où le masque peut manquer une opacité.

**P7 : le format d'origine compte-t-il encore ?** Une dépendance plus faible en v4 montre que le recadrage retire une partie de l'information de format. Regarder aussi `diagnostic_entrees_v4.csv` : si la géométrie de la boîte sépare bien les classes, le réseau dispose d'un nouvel indice indirect.

**P8 : le prix du recadrage.** Une perte de F1 de validation limitée à 0,02 rend la v4 acceptable. Une perte plus grande peut aussi vouloir dire que la v3 profitait encore d'indices hors des poumons, présents dans la validation.

**CBAM contre classique en v4.** C'est la comparaison la plus propre du mémoire : les deux architectures voient la même zone thoracique. Un CBAM plus concentré dans les poumons, même sans gain de F1, est un argument d'interprétabilité. L'attention spatiale du CBAM (dans et hors des poumons) montre si le module d'attention lui-même privilégie les poumons.

**Test exploratoire.** Il répond à la question qui compte pour l'application clinique : le recadrage réduit-il les faux positifs sur les radios NORMAL du test, surtout les plus larges ? Ces résultats s'écrivent au conditionnel (« analyse exploratoire, après l'évaluation principale ») et ne remplacent pas H1 à H4.

**Limites.** Masques non vérifiés par un radiologue ; segmenteur conçu pour l'adulte ; Grad-CAM reste une approximation à 28 × 28 ; trois graines ; la validation a servi à choisir les époques.

**Suite :** envoyer `a_envoyer_notebook_16.zip`. Les masques enregistrés dans `masques/` serviront, si besoin, au deuxième levier : une perte qui guide l'attention vers les poumons.